# Cas pratique - Campagne de démarchage bancaire

## 0 - Imports

In [ ]:
# ============================================================
# Imports
# ============================================================

# Bibliothèque standard
import sys
import time
import subprocess
from datetime import datetime
from pathlib import Path

# Manipulation / visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn — préparation
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Scikit-learn — modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)

# Scikit-learn — validation
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    cross_validate,
    cross_val_predict,
    StratifiedKFold,
)

# Scikit-learn — métriques
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Scikit-learn — interprétabilité
from sklearn.inspection import permutation_importance

# Persistance
import joblib


# ============================================================
# Configuration globale
# ============================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 0.5 Traçabilité de la session

In [ ]:
# --- Métadonnées du dataset ---
DATASET_NAME = "Bank Marketing"
DATASET_SOURCE = "../data/bank-additional-full.csv"
DATASET_VERSION = "YYYY-MM-DD"  # date d'extraction OU version explicite

# --- Métadonnées de la session ---
session_date = datetime.now().isoformat(timespec="minutes")
py_version = sys.version.split()[0]

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    git_commit = "non disponible (notebook hors repo Git)"

print(f"Date session : {session_date}")
print(f"Python       : {py_version}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Git commit   : {git_commit}")
print(f"Dataset      : {DATASET_NAME} (source={DATASET_SOURCE}, version={DATASET_VERSION})")

# 1. Cadrage métier & problème

## 1.1 Contexte client

Le client est une banque de détail souhaitant améliorer l'efficacité de ses campagnes de démarchage téléphonique pour la souscription à des dépôts à terme.

Pour proposer ce type de produit d'épargne, les banques mènent régulièrement des campagnes de démarchage téléphonique.

Ces campagnes sont coûteuses et peuvent dégrader la relation client lorsqu'elles sont mal ciblées.

Le besoin métier est donc de réduire le nombre d'appels inutiles et le coût des campagnes, tout en augmentant le taux de conversion des offres.

Le recours à l'IA doit permettre d'identifier les clients les plus susceptibles de souscrire à partir de leur profil et de l'historique de leurs interactions avec la banque.


## 1.2 Énoncé du problème IA

Le problème correspond à une tâche de classification binaire.

L'objectif du modèle est de prédire, à partir des informations disponibles sur un client, si celui-ci est susceptible de souscrire à un dépôt à terme (`yes`) ou non (`no`).

La solution devra permettre d'effectuer cette prédiction avant la prise de contact afin d'aider au ciblage des campagnes.


## 1.3 Bénéficiaires & impacts

Les bénéficiaires directs de la solution sont :
- la banque ;
- les équipes chargées des campagnes commerciales, qui pourront mieux cibler les clients à contacter.

Les bénéficiaires indirects sont les clients de la banque, qui pourront être moins sollicités pour des offres qui ne les intéressent pas.

Les prédictions du modèle peuvent cependant avoir un impact sur les clients en déterminant quels profils seront davantage ou moins sollicités.


## 1.4 Critères de succès

| Type | Critère | Cible initiale (client) | Cible révisée après EDA | Justification de la révision |
|---|---|---|---|---|
| Métier | Réduire le nombre d'appels inutiles | Améliorer le ciblage des campagnes | Réduire les faux positifs | Chaque faux positif peut conduire à une sollicitation inutile |
| Métier | Augmenter le taux de conversion | Augmenter la proportion de souscriptions parmi les clients contactés | Maximiser la précision sans rendre le rappel inutilisable | La précision correspond directement à la fiabilité du ciblage |
| Modèle | Identifier les clients susceptibles de souscrire | Non chiffré par le client | Suivre précision, rappel, F1 et ROC-AUC | La classe positive est minoritaire, l'accuracy seule serait trompeuse |
| Opérationnel | Prédiction avant prise de contact | Prédiction disponible avant l'appel | Exclure toute variable indisponible avant l'appel | `duration` n'est connue qu'après l'appel |
| Réglementaire / éthique | Limiter les risques liés aux données personnelles | Respect du cadre applicable | Réduire l'utilisation directe des variables socio-démographiques | Les écarts entre groupes observés pendant l'EDA justifient une surveillance spécifique |

Les valeurs chiffrées des objectifs modèle seront précisées après l'analyse exploratoire des données.


## 1.5 Risques éthiques & réglementaires anticipés

Plusieurs risques devront être pris en compte :

- utilisation de données personnelles pour du ciblage commercial
- risque de biais ou de traitement différencié entre certains groupes de clients
- utilisation de variables qui ne seraient pas disponibles au moment réel de la prédiction
- manque d'explicabilité du modèle

Le traitement repose sur des données personnelles. Il devra donc respecter le RGPD, notamment les principes de minimisation des données, de limitation de la finalité et de transparence.

L'AI Act devra également être pris en compte afin de déterminer les obligations applicables à la solution.

### 1.5.1 Protection des données personnelles

Dans un contexte réel de déploiement bancaire, les données mobilisées sont des
données à caractère personnel dès lors qu'elles sont rattachables à un client
identifié ou identifiable.

Sont notamment concernées :

- les informations socio-démographiques (`age`, `job`, `marital`, `education`)
- les informations relatives à la situation bancaire (`default`, `housing`, `loan`)
- les informations relatives aux contacts et campagnes précédentes
- les scores et probabilités produits par le modèle lorsqu'ils sont associés à un client

Les variables présentes dans le jeu de données ne relèvent pas, en l'état de leur
description, des catégories particulières de données visées à l'article 9 du RGPD
(par exemple données de santé, opinions politiques, religion, biométrie ou
orientation sexuelle).

Certaines variables peuvent néanmoins être sensibles dans un sens métier ou
éthique sans être des données relevant juridiquement de l'article 9.

Pour un traitement de prospection commerciale, une base légale envisageable est
l'intérêt légitime du responsable de traitement, sous réserve de vérifier que cet
intérêt ne porte pas une atteinte disproportionnée aux droits et libertés des
personnes et de respecter les règles spécifiques applicables à la prospection.

Cette base légale et les modalités de prospection devraient être validées avec le
DPO ou le service juridique avant mise en production.

Les principales obligations à prendre en compte sont :

- information claire des personnes sur l'utilisation de leurs données à des fins
  de ciblage commercial
- respect du droit d'opposition à la prospection
- minimisation des données : ne conserver et n'utiliser que les variables utiles
  au ciblage
- limitation de la finalité : ne pas réutiliser ces données pour un objectif
  incompatible sans nouvelle analyse
- limitation de la durée de conservation : définir une durée adaptée à la
  campagne et aux obligations internes, puis supprimer ou anonymiser les données
  devenues inutiles
- contrôle des accès et traçabilité des traitements

Le scénario S3 contribue au principe de minimisation en excluant `age`, `job`,
`marital`, `education` et `duration` du modèle final. Cette réduction n'élimine
cependant pas le risque de biais indirect, qui est évalué séparément par
sous-groupe.

### 1.5.2 Qualification au regard de l'AI Act

L'usage étudié consiste à prioriser les clients à contacter dans le cadre d'une
campagne commerciale portant sur un dépôt à terme.

Le système ne détermine pas la solvabilité d'un client, n'établit pas de score
de crédit et ne décide pas de son accès à un service financier.

Dans cet usage précis, il ne relève donc pas des cas d'usage à haut risque de
l'annexe III liés à l'évaluation de la solvabilité ou au credit scoring.

Cette qualification dépend toutefois de la finalité du système et devrait être
réévaluée si le modèle était réutilisé pour une décision de crédit, de tarification
ou d'accès à un service financier.

## 1.6 Synthèse cadrage

Le projet vise à améliorer le ciblage des campagnes téléphoniques d'une banque de détail afin de réduire les sollicitations inutiles et d'améliorer le taux de conversion.

La solution reposera sur un modèle de classification binaire prédisant si un client est susceptible de souscrire à un dépôt à terme.

La réussite du projet devra être évaluée à la fois selon des critères métier, techniques et opérationnels.

Le principal point de vigilance concerne l'utilisation de données personnelles et le risque de traitement différencié entre certains groupes de clients.

# 2. Identification & acquisition des données

## 2.1 Sources

Le jeu de données utilisé est `bank-additional-full.csv`, issu du dataset public **Bank Marketing** du UCI Machine Learning Repository.

Il contient 41 188 observations, 20 variables explicatives et une variable cible `y`.

| Source | Type | Volumétrie | Format | Accès | Date extraction |
|---|---|---:|---|---|-----------------|
| UCI Machine Learning Repository – Bank Marketing | Jeu de données public | 41 188 lignes | CSV, séparateur `;` | Fichier fourni dans le cadre du projet | Non Documentée  |

**Licence :** Creative Commons Attribution 4.0 International — CC BY 4.0

## 2.2 Chargement

In [ ]:
df = pd.read_csv(DATASET_SOURCE, sep=";")


## 2.3 Dictionnaire de variables

| Variable | Description | Type | Plage / valeurs | Cible ? | Sensible ? |
|---|---|---|---|---|---|
| `age` | Âge du client | Numérique | Valeur numérique | ❌ | Donnée personnelle socio-démographique, hors art. 9 RGPD, vigilance sur les écarts de traitement |
| `job` | Type d'emploi | Catégorielle | admin., blue-collar, entrepreneur, housemaid, management, retired, self-employed, services, student, technician, unemployed, unknown | ❌ | Donnée personnelle socio-professionnelle, hors art. 9 RGPD, risque de biais indirect |
| `marital` | Statut marital | Catégorielle | married, single, divorced, unknown | ❌ | Donnée personnelle relative à la situation familiale, hors art. 9 RGPD |
| `education` | Niveau d'éducation | Catégorielle | basic.4y, basic.6y, basic.9y, high.school, illiterate, professional.course, university.degree, unknown | ❌ | Donnée personnelle socio-éducative, hors art. 9 RGPD, risque de biais indirect |
| `default` | Crédit en défaut de paiement | Catégorielle | yes, no, unknown | ❌ | Donnée personnelle financière, vigilance élevée |
| `housing` | Prêt immobilier | Catégorielle | yes, no, unknown | ❌ | Donnée personnelle financière |
| `loan` | Prêt personnel | Catégorielle | yes, no, unknown | ❌ | Donnée personnelle financière |
| `contact` | Moyen de contact | Catégorielle | cellular, telephone | ❌ | ❌ |
| `month` | Mois du dernier contact | Catégorielle | jan à dec | ❌ | ❌ |
| `day_of_week` | Jour du dernier contact | Catégorielle | mon, tue, wed, thu, fri | ❌ | ❌ |
| `duration` | Durée du dernier contact en secondes | Numérique | Valeur numérique | ❌ | ❌ |
| `campaign` | Nombre de contacts pendant la campagne | Numérique | Valeur numérique | ❌ | ❌ |
| `pdays` | Jours depuis le dernier contact d'une campagne précédente | Numérique | 999 = jamais recontacté | ❌ | ❌ |
| `previous` | Nombre de contacts avant cette campagne | Numérique | Valeur numérique | ❌ | ❌ |
| `poutcome` | Résultat de la campagne précédente | Catégorielle | failure, nonexistent, success | ❌ | ❌ |
| `emp.var.rate` | Taux de variation de l'emploi | Numérique | Indicateur trimestriel | ❌ | ❌ |
| `cons.price.idx` | Indice des prix à la consommation | Numérique | Indicateur mensuel | ❌ | ❌ |
| `cons.conf.idx` | Indice de confiance des consommateurs | Numérique | Indicateur mensuel | ❌ | ❌ |
| `euribor3m` | Taux Euribor à 3 mois | Numérique | Indicateur journalier | ❌ | ❌ |
| `nr.employed` | Nombre de salariés | Numérique | Indicateur trimestriel | ❌ | ❌ |
| `y` | Souscription à un dépôt à terme | Catégorielle binaire | yes / no | ✅ | ❌ |

## 2.4 Synthèse acquisition

Le projet repose sur un jeu de données public issu du UCI Machine Learning Repository.

Il contient 41 188 observations, 20 variables explicatives et une cible binaire indiquant la souscription ou non à un dépôt à terme.

Les données couvrent plusieurs dimensions : profil socio-démographique, situation bancaire, historique des campagnes, informations sur le dernier contact et contexte socio-économique.

Plusieurs variables liées au profil du client devront faire l'objet d'une attention particulière lors de l'analyse éthique et réglementaire.

La qualité réelle des données sera évaluée lors de l'analyse exploratoire.

# 3. Exploration & analyse des données (EDA)

L'objectif de cette phase est de comprendre la structure et la qualité des données avant toute transformation ou modélisation.

L'analyse portera notamment sur :
- la qualité des données
- la distribution des variables
- la répartition de la cible
- les relations entre les variables et la cible
- les éventuels risques de biais liés aux variables disponibles

## 3.1 Premier coup d'œil

Nous commençons par examiner la structure générale du jeu de données, les types
de variables et quelques statistiques descriptives.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 3.2 Qualité des données

In [ ]:
print("Missing values (NaN):")
print(df.isna().sum())

print("\nUnknown values:")
print((df == "unknown").sum())

print("Nombre de doublons :", df.duplicated().sum())

Aucune valeur manquante au format `NaN` n'est présente dans le jeu de données.

Certaines variables catégorielles contiennent cependant une modalité `unknown`.
Cette modalité est particulièrement présente pour `default`, tandis qu'elle
reste plus limitée pour les autres variables.

12 lignes strictement identiques sont également présentes. En l'absence
d'identifiant client permettant de déterminer s'il s'agit de doublons techniques
ou de clients différents présentant les mêmes caractéristiques, elles sont
conservées à ce stade de l'analyse.

## 3.3 Distribution des variables

L'objectif ici est d'identifier les caractéristiques de leur distribution qui pourraient influencer
la préparation des données et la modélisation.

On va d'abord chercher dans les variables numériques une distribution atypique :

In [ ]:
numeric_values = ["age", "duration", "campaign", "pdays", "previous"]

for col in numeric_values:
    df[col].hist(bins=30)
    plt.title(f"Distribution de {col}")
    plt.xlabel(col)
    plt.ylabel("Nombre d'observations")
    plt.show()

Puis chercher un recoupement avec les souscriptions :

In [ ]:
variables_to_compare = ["age", "duration", "campaign", "previous"]

for col in variables_to_compare:
    df.boxplot(column=col, by="y")
    plt.title(f"{col} selon la souscription")
    plt.suptitle("")
    plt.show()

### Cas particulier : `pdays`

`pdays` utilise la valeur `999` pour représenter un client qui n'avait jamais été
contacté auparavant. Une comparaison directe par boxplot mélangerait donc une
valeur codée avec de véritables nombres de jours, et ne serait pas pertinente.

In [ ]:
df[df["pdays"] != 999].boxplot(column="pdays", by="y")
plt.title("pdays selon la souscription chez les clients déjà contactés")
plt.suptitle("")
plt.show()

Nous allons ensuite chercher si certaines variables catégorielles sont rares ou dominantes :

In [ ]:
categorical_values = df.select_dtypes(include="object").columns.drop("y")

for col in categorical_values:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(normalize=True).mul(100).round(2))

### Constats sur les variables catégorielles

L'analyse des fréquences montre plusieurs distributions déséquilibrées.

Certaines modalités sont très peu représentées, notamment `illiterate` pour
`education` et `yes` pour `default`, ce qui pourra rendre leur analyse statistique
moins robuste.

La variable `default` se distingue également par une proportion importante
de valeurs `unknown`.

`poutcome` est très majoritairement égal à `nonexistent`, indiquant que la plupart
des clients ne disposent pas d'un résultat exploitable de campagne précédente.

À l'inverse, `day_of_week` présente une distribution relativement homogène.

## 3.4 Variable cible : équilibre des classes

In [ ]:
target_counts = df["y"].value_counts()
target_rates = df["y"].value_counts(normalize=True).mul(100).round(2)

print("Effectifs :")
print(target_counts)

print("\nProportions :")
print(target_rates.astype(str) + "%")

In [ ]:
target_rates.plot(kind="bar")

plt.ylabel("Proportion (%)")
plt.title("Répartition de la variable cible")
plt.xticks(rotation=0)
plt.show()

La variable cible est fortement déséquilibrée :

- `no` représente 88,73 % des observations ;
- `yes` représente 11,27 % des observations.

La classe correspondant à la souscription est donc minoritaire.

Ce déséquilibre devra être pris en compte lors de l'évaluation des modèles.
L'accuracy seule serait insuffisante : un modèle prédisant systématiquement
`no` obtiendrait déjà une accuracy de 88,73 % sans permettre d'identifier
les clients susceptibles de souscrire.

Les performances devront donc également être évaluées à l'aide de métriques
adaptées à la classe positive, notamment la précision, le rappel, le F1-score,
la ROC-AUC et la matrice de confusion.

## 3.5 Corrélations & relations entre variables

L'objectif de cette section est d'identifier :
- d'éventuelles redondances entre variables numériques
- les variables présentant les relations les plus visibles avec la variable cible.

In [ ]:
numeric_df = df.select_dtypes(include="number")

corr = numeric_df.corr()

corr

### Corrélations entre variables numériques

La matrice de corrélation met en évidence plusieurs relations fortes entre les variables macro-économiques.

En particulier, `emp.var.rate`, `euribor3m` et `nr.employed` sont fortement corrélées entre elles, ce qui suggère qu'elles véhiculent en partie une information redondante sur le contexte économique.

À l'inverse, les variables liées directement au client ou à la campagne, comme `age`, `duration` ou `campaign`, présentent globalement de faibles corrélations linéaires avec les autres variables numériques.

Ces corrélations ne justifient pas à elles seules la suppression de variables, mais elles devront être prises en compte lors de la modélisation.

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")

plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=90
)

plt.yticks(
    range(len(corr.columns)),
    corr.columns
)

plt.colorbar()
plt.title("Matrice de corrélation des variables numériques")
plt.show()

Après avoir étudié la répartition propre de chaque variable catégorielle, nous cherchons maintenant à mesurer leur relation avec la variable cible.

Pour chaque modalité, nous calculons donc la proportion de clients ayant souscrit afin d’identifier d’éventuelles différences de taux de souscription entre les groupes.

In [ ]:
categorical_to_compare = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome"
]

for col in categorical_to_compare:
    rates = (
        df.groupby(col)["y"]
        .apply(lambda x: (x == "yes").mean())
        .sort_values(ascending=False)
        .mul(100)
        .round(2)
    )

    print(f"\n--- {col} ---")
    print(rates)

In [ ]:
for col in ["job", "education", "contact", "poutcome"]:
    rates = (
        df.groupby(col)["y"]
        .apply(lambda x: (x == "yes").mean())
        .sort_values(ascending=False)
        .mul(100)
    )

    rates.plot(kind="bar")
    plt.ylabel("Taux de souscription (%)")
    plt.title(f"Taux de souscription selon {col}")
    plt.xticks(rotation=45)
    plt.show()

## 3.6 Analyse des biais & variables sensibles

Certaines variables décrivent directement le profil socio-démographique du client et
peuvent entraîner des différences de traitement entre groupes, notamment :

- `age`
- `job`
- `marital`
- `education`

À ce stade, avant l'entraînement d'un modèle, nous pouvons vérifier si la variable cible
est répartie différemment entre ces groupes et si certains d'entre eux sont très peu
représentés.

L'analyse des biais liée aux prédictions du modèle (taux de sélection, faux positifs,
faux négatifs, disparate impact) sera complétée après la phase de modélisation.

In [ ]:
bias_variables = ["job", "marital", "education"]

for col in bias_variables:
    rates = (
        df.groupby(col)["y"]
        .apply(lambda x: (x == "yes").mean())
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
    )

    print(f"\n--- Taux de souscription selon {col} ---")
    print(rates)

In [ ]:
age_group = pd.cut(
    df["age"],
    bins=[0, 30, 40, 50, 60, float("inf")],
    labels=["<30", "30-39", "40-49", "50-59", "60+"]
)

age_subscription_rate = (
    df.assign(age_group=age_group)
    .groupby("age_group", observed=False)["y"]
    .apply(lambda x: (x == "yes").mean())
    .mul(100)
    .round(2)
)

age_subscription_rate

Plusieurs variables socio-démographiques présentent des différences de taux de souscription entre les groupes.

La profession (`job`) montre des écarts importants : certaines catégories comme les étudiants ou les retraités présentent un taux de souscription nettement supérieur à d'autres, comme les ouvriers (`blue-collar`).

Le niveau d'éducation présente également des écarts visibles, mais certaines modalités sont très peu représentées, ce qui limite la robustesse des comparaisons.

L'âge montre une différence particulièrement marquée : les clients de 60 ans et plus présentent un taux de souscription beaucoup plus élevé que les autres tranches d'âge.

Ces écarts ne démontrent pas à eux seuls l'existence d'un biais du modèle, puisqu'aucun modèle n'a encore été entraîné. Ils montrent toutefois que certaines variables sensibles ou socio-démographiques sont fortement associées à la cible et devront faire l'objet d'une attention particulière lors de la modélisation et de l'évaluation de l'équité.

L'analyse des biais sur les prédictions du modèle, notamment les taux de sélection, faux positifs, faux négatifs et éventuel disparate impact, sera réalisée après entraînement.

## 3.7 Synthèse EDA

L'analyse exploratoire met en évidence plusieurs éléments importants pour la suite du projet :

1. La qualité générale des données est bonne : aucune valeur `NaN` n'est présente. Certaines variables contiennent toutefois la modalité `unknown`, particulièrement `default`.

2. La variable cible est fortement déséquilibrée : seulement 11,27 % des observations correspondent à une souscription. L'accuracy seule ne sera donc pas suffisante pour évaluer les modèles.

3. Certaines variables présentent des particularités nécessitant un traitement ou une interprétation spécifique, notamment `pdays`, dont la valeur `999` signifie que le client n'avait jamais été contacté auparavant.

4. Plusieurs variables macro-économiques, notamment `emp.var.rate`, `euribor3m` et `nr.employed`, sont fortement corrélées entre elles, ce qui suggère une redondance potentielle.

5. Plusieurs variables socio-démographiques présentent des différences importantes de taux de souscription selon les groupes, en particulier `age` et `job`. Elles devront donc être analysées avec attention afin de limiter les risques de biais ou de traitement différencié.

Enfin, `duration` semble fortement liée à la cible, mais cette information n'est disponible qu'après la fin de l'appel. Elle ne pourra donc pas être utilisée directement dans un modèle destiné à décider en amont quels clients contacter.

### Conséquences pour la modélisation

La suite du projet devra :
- utiliser des métriques adaptées à la classe minoritaire, notamment précision, rappel et F1-score ;
- comparer les performances avec et sans `duration` ;
- analyser l'impact des variables socio-démographiques ;
- contrôler les performances du modèle par sous-groupes afin d'identifier d'éventuels biais.

# 4. Préparation des données

Nous allons maintenant préparer les données pour l'entraînement du modèle.

Le découpage entre données d'entraînement et de test est réalisé avant tout apprentissage du préprocessing. Les transformations seront ensuite intégrées dans un `Pipeline` afin qu'elles soient ajustées uniquement sur les données d'entraînement.

In [ ]:
X = df.drop(columns=["y"])
y = df["y"].map({"no": 0, "yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train :", X_train.shape)
print("Test :", X_test.shape)

print("\nRépartition cible train :")
print(y_train.value_counts(normalize=True).round(4))

print("\nRépartition cible test :")
print(y_test.value_counts(normalize=True).round(4))

Le jeu de données est séparé en 80 % d'entraînement et 20 % de test.

Le paramètre `stratify=y` permet de conserver une proportion similaire de clients ayant souscrit dans les deux ensembles, ce qui est particulièrement important compte tenu du déséquilibre de la cible observé lors de l'EDA.

In [ ]:
def make_preprocessor(num_features=None, cat_features=None):
    blocks = []

    if num_features:
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])

        blocks.append(("num", num_pipe, num_features))

    if cat_features:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])

        blocks.append(("cat", cat_pipe, cat_features))

    assert blocks, "Aucune colonne sélectionnée pour le preprocessing"

    return ColumnTransformer(blocks)

Les variables numériques manquantes seront imputées par la médiane puis standardisées.
La médiane est choisie car moins sensible aux valeurs extrêmes que la moyenne, et comme on a pu le constater lors de l'EDA, certaines variables numériques présentent des distributions asymétriques.

Pour les variables catégorielles, les valeurs manquantes seront imputées par la modalité `missing` puis encodées en One-Hot, ce qui permet de conserver l'information de la modalité manquante plutot que de les assimiler a une autre catégorie artificiellement.
`handle_unknown="ignore"` permet au modèle de traiter une modalité apparaissant en production mais absente des données d'entraînement sans provoquer d'erreur, dans le cas ou par exemple une nouvelle modalité inconnue à l'entrainement serait présente.

Aucun `fit` n'est réalisé à cette étape. Le préprocessing sera entraîné uniquement au sein du pipeline complet de modélisation.

## 4.3 Scénarios de jeux de données

Afin de mesurer l'impact des variables problématiques identifiées lors de l'EDA,
plusieurs périmètres de données seront comparés.

| Scénario | Variables conservées | Justification |
|---|---|---|
| S1 — Complet | Toutes les variables, y compris `duration` | Référence de performance |
| S2 — Sans `duration` | Toutes sauf `duration` | Éliminer une variable indisponible avant la fin de l'appel |
| S3 — Sans `duration` ni variables socio-démographiques sensibles | Retrait de `duration`, `age`, `job`, `marital`, `education` | Mesurer le compromis entre performance et réduction des risques éthiques |
| S4 — Minimal (optionnel) | S3 sans les variables liées à l'historique des campagnes | Tester la faisabilité d'un modèle reposant sur un périmètre de données réduit |

In [ ]:
sensitive_features = [
    "age",
    "job",
    "marital",
    "education"
]

campaign_history_features = [
    "campaign",
    "pdays",
    "previous",
    "poutcome"
]

scenarios = {
    "S1_all": X.columns.tolist(),

    "S2_without_duration": [
        col for col in X.columns
        if col != "duration"
    ],

    "S3_without_duration_or_sensitive": [
        col for col in X.columns
        if col != "duration"
           and col not in sensitive_features
    ],

    "S4_minimal": [
        col for col in X.columns
        if col != "duration"
           and col not in sensitive_features
           and col not in campaign_history_features
    ]
}

for name, features in scenarios.items():
    print(name, len(features), "variables")

In [ ]:
def get_feature_types(X, features):
    scenario_df = X[features]

    num_features = (
        scenario_df
        .select_dtypes(include="number")
        .columns
        .tolist()
    )

    cat_features = (
        scenario_df
        .select_dtypes(include="object")
        .columns
        .tolist()
    )

    return num_features, cat_features

## 4.4 Synthèse préparation

Les données ont été séparées en ensembles d'entraînement et de test avec stratification de la variable cible afin de préserver son déséquilibre initial.

Une fonction de preprocessing reproductible a été définie à partir d'un `ColumnTransformer` :
- standardisation des variables numériques ;
- encodage One-Hot des variables catégorielles ;
- gestion d'éventuelles valeurs manquantes.

Quatre scénarios de données ont été définis afin de mesurer séparément l'impact de `duration`, des variables socio-démographiques et de l'historique de campagne.

Aucune transformation n'est ajustée à ce stade : le `fit` du preprocessing sera effectué exclusivement sur les données d'entraînement au sein des pipelines de modélisation.

## 4.5 Assertions de qualité

In [ ]:
assert df.shape[0] > 0, "Le dataset est vide"

assert y.isna().sum() == 0, "La cible contient des valeurs manquantes"

assert set(y.unique()) == {0, 1}, "La cible doit être binaire"

assert (
               X_train.shape[0] + X_test.shape[0]
       ) == df.shape[0], "Le split train/test est incomplet"

assert set(X_train.columns) == set(X_test.columns), \
    "Les colonnes du train et du test sont différentes"

assert abs(y_train.mean() - y_test.mean()) < 0.01, \
    "La stratification n'a pas conservé correctement la distribution de la cible"

for name, features in scenarios.items():
    assert len(features) > 0, f"{name} ne contient aucune variable"
    assert len(features) == len(set(features)), \
        f"{name} contient des variables dupliquées"

assert "duration" in scenarios["S1_all"]

assert "duration" not in scenarios["S2_without_duration"]

assert "duration" not in scenarios["S3_without_duration_or_sensitive"]

assert all(
    feature not in scenarios["S3_without_duration_or_sensitive"]
    for feature in sensitive_features
)

# 5 Modélisation & entrainement

## 5.0 Familles de modèles candidates

| Famille                           | Adapté à mon problème ?                                                         | Coût total | Explicabilité                    | Dépendance fournisseur | Décision                                                                                          |
| --------------------------------- | ------------------------------------------------------------------------------- | ---------- | -------------------------------- | ---------------------- | ------------------------------------------------------------------------------------------------- |
| ML classique (`sklearn`, XGBoost) | Oui : classification binaire sur données tabulaires structurées, volume modéré  | Faible     | Élevée à moyenne selon le modèle | Aucune                 | **Retenue** — adaptée au type de données, peu coûteuse, facilement industrialisable et explicable |
| Deep Learning (Keras / PyTorch)   | Peu pertinent : pas d’image, texte ou signal complexe ; volume limité           | Élevé      | Faible                           | Aucune si local        | **Écartée** — complexité et coût non justifiés par le cas d’usage                                 |
| SLM local                         | Non pertinent : pas de génération de texte ni de compréhension sémantique       | Moyen      | Moyenne à faible                 | Aucune si local        | **Écartée** — ne correspond pas à la nature du problème                                           |
| LLM API + RAG                     | Non pertinent : pas de corpus documentaire ni de question ouverte               | Élevé      | Faible                           | Forte                  | **Écartée** — surdimensionné, coûteux et inutile pour une classification tabulaire                |
| Architecture agentique            | Non pertinent : aucune orchestration multi-étapes ou action autonome nécessaire | Très élevé | Très faible                      | Forte                  | **Écartée** — complexité sans bénéfice pour le besoin                                             |


## 5.0.1 Synthèse familles

Les modèles de ML classiques sont les plus adaptés à ce type de problème : Classification binaire sur données structurées, volume modéré (~40k lignes), besoin d'explicabilité et de reproductibilité.

Les modèles de Deep Learning seraient plus couteux et moins explicables, sans bénéfice réel dans notre cas puisque nous ne traitons que des données structurées. Ils auraient été envisagés sur un volume de données massif, ou si nous avions des données additionnelles non structurées comme des images ou des textes.

Les modèles de SLM local n'apportent pas de solution à notre problème, puisqu'ici on ne se préoccupe ni de compréhension de texte ni de génération de texte.

Les LLM API + RAG ne sont pas non plus adaptés à notre cas, puisque nous n'avons pas de corpus documentaire à traiter, seraient couteux et ne permettraient pas d'obtenir un modèle explicable.

Une architecture agentique n'apporterait pas de bénéfice supplémentaire pour notre problème, étant donné l'absence de besoin d'orchestration multi-étapes ou d'actions autonomes.

## 5.1 Modèles candidats

| Modèle                | Famille                      | Pourquoi candidat ?                                                                         | Coût entraînement / inférence | Explicabilité |
| --------------------- | ---------------------------- | ------------------------------------------------------------------------------------------- | ----------------------------- | ------------- |
| Régression logistique | Linéaire                     | Baseline simple, rapide et facilement interprétable                                         | Faible / faible               | Élevée        |
| Random Forest         | Ensemble d'arbres            | Capable de modéliser des relations non linéaires, robuste et adaptée aux données tabulaires | Moyen / faible                | Moyenne       |
| Gradient Boosting     | Ensemble séquentiel d'arbres | Bon candidat pour améliorer les performances sur des données tabulaires structurées         | Moyen à élevé / faible        | Moyenne       |



## 5.2 Entrainement & validation croisée

Chaque modèle sera évalué sur l'ensemble des scénarios prévus à la section 4.3.

Pour garantir la comparabilité des résultats, la recette de préprocessing sera identique pour tous les scénarios, et seuls le périmètre des variables et le modèle changeront.

L'entrainement et le préprocessing seront réalisés dans un pipeline unique, afin d'éviter tout risque de fuite de données entre l'entraînement et la validation.

In [ ]:
modeles = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    "RandomForestClassifier": RandomForestClassifier(
        random_state=RANDOM_STATE,
    ),
    "GradientBoostingClassifier": GradientBoostingClassifier(
        random_state=RANDOM_STATE,
    )
}

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [ ]:
scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

results = []

for scenario, features in scenarios.items():
    num_features, cat_features = get_feature_types(
        X_train,
        features
    )

    for model_name, model in modeles.items():
        preprocessor = make_preprocessor(
            num_features=num_features,
            cat_features=cat_features,
        )

        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        scores = cross_validate(
            pipe,
            X_train[features],
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        )

        results.append({
            "scenario": scenario,
            "model": model_name,
            "precision_mean": scores["test_precision"].mean(),
            "recall_mean": scores["test_recall"].mean(),
            "f1_mean": scores["test_f1"].mean(),
            "roc_auc_mean": scores["test_roc_auc"].mean(),
            "f1_std": scores["test_f1"].std(),
        })

## 5.3 Métriques retenues et justification

La variable cible étant très déséquilibrée, l'accuracy ne peut pas être utilisée seule pour évaluer les performances des modèles.

Ensuite, le client à pour objectif de réduire les appels inutiles (nuisance pour les clients) et de maximiser le taux de conversion sur un nombre d'appels réduit.

Nous avons donc plusieurs angles à considérer :
- On veut éviter de contacter des clients qui ne souscriront pas, donc on veut maximiser la précision
- On veut éviter de rater des clients qui souscriront, donc on veut maximiser le rappel

La comparaison en validation croisée s'appuie principalement sur le F1-score de la classe positive puisque celui-ci va combiner :
- La précision, c'est à dire les vrais positifs parmi les prédictions positives (parmi les clients prédits comme futurs souscripteurs, combien le sont réellement devenus))
- Le rappel, c'est à dire les vrais positifs parmi les positifs réels (parmi les clients ayant réellement souscrit, combien ont été correctement identifiés par le modèle)

Cette métrique permet donc de pénaliser simultanément les faux positifs et les faux négatifs à la même échelle, ce qui est cohérent avec les objectifs du projet.

Pour l'évaluation détaillée des modèles, nous comparerons également :
- Précision
- Rappel
- ROC-AUC
- Matrice de confusion


## 5.4 Comparaison des modèles : tableau de scores CV

In [ ]:
cv_results = pd.DataFrame(results)

cv_results.sort_values(
    by="f1_mean",
    ascending=False
)

Les modèles ont été comparés par validation croisée sur les quatre scénarios définis précédemment.

Le scénario S1, qui inclut toutes les variables et notamment `duration`, obtient les meilleurs scores pour tous les modèles, ce qui est cohérent avec le fait que `duration` est fortement corrélée à la cible.

Si on compare maintenant les scénarios utilisant uniquement les données pré-appel, les résultats font apparaitre de nouvelles informations :
- Le Random Forest sur le scénario S3 obtient le meilleur F1-score (0.387) et le meilleur rappel (0.306)
- Le Gradient Boosting sur le scénario S3 obtient la meilleure précision (0.673) et la meilleure ROC-AUC (0.796)
- La Logistic Regression obtient également une précision élevée, mais avec un rappel plus faible
- Le scénario S4 entraine globalement une diminution des performances, en particulier sur la Logistic Regression

Retirer des variables démographiques entre les scénarios S2 et S3 n'entraine pas de perte de performance significative, ce qui est encourageant pour réduire les risques éthiques et réglementaires.

Ce scénario sera donc notre scénario d'évaluation principal, puisqu'il permet de maximiser les performances tout en limitant les risques liés aux variables sensibles.

Nous pouvons donc conclure qu'un choix de modèle basé uniquement sur le F1-score ne serait pas suffisant, et qu'il faudra considérer les autres métriques pour faire un choix final.

Ainsi, si la banque ne peut se permettre d'appeler qu'un nombre limité de clients, on privilégiera le Gradient Boosting pour maximiser la précision malgré un F1-score inférieur.

## 5.4.1 Mesures opérationnelles

Afin de compléter la comparaison des scénarios par des critères opérationnels, les modèles représentatifs sont également comparés en termes de temps d'inférence.

Les mesures sont réalisées dans les mêmes conditions et uniquement sur les données d'entrainement, afin de ne conserver le jeu de test vierge pour l'évaluation finale.

In [ ]:
scenario_models = {
    "S1_all": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    "S2_without_duration": RandomForestClassifier(
        random_state=RANDOM_STATE
    ),
    "S3_without_duration_or_sensitive": RandomForestClassifier(
        random_state=RANDOM_STATE
    ),
    "S4_minimal": RandomForestClassifier(
        random_state=RANDOM_STATE
    ),
}

In [ ]:
benchmark_results = []

repeats = 50

for scenario, features in scenarios.items():

    num_features, cat_features = get_feature_types(
        X_train,
        features
    )

    for model_name, model in modeles.items():

        preprocessor = make_preprocessor(
            num_features=num_features,
            cat_features=cat_features,
        )

        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipe.fit(
            X_train[features],
            y_train
        )

        X_bench = X_train[features].sample(
            n=min(1000, len(X_train)),
            random_state=RANDOM_STATE
        )

        # Warm-up
        pipe.predict_proba(X_bench.iloc[:10])

        # Latence unitaire
        latencies = []

        for i in range(len(X_bench)):
            row = X_bench.iloc[[i]]

            start = time.perf_counter()
            pipe.predict_proba(row)
            latencies.append(
                (time.perf_counter() - start) * 1000
            )

        # CPU sur un batch de 1000, répété pour stabiliser la mesure
        cpu_start = time.process_time()

        for _ in range(repeats):
            pipe.predict_proba(X_bench)

        cpu_seconds = time.process_time() - cpu_start

        cpu_s_per_1000 = (
            cpu_seconds
            / repeats
            * (1000 / len(X_bench))
        )

        benchmark_results.append({
            "scenario": scenario,
            "model": model_name,
            "latency_mean_ms": np.mean(latencies),
            "latency_p95_ms": np.percentile(latencies, 95),
            "cpu_s_per_1000": cpu_s_per_1000,
        })

benchmark_results = pd.DataFrame(benchmark_results)

benchmark_results.round(3)

Les mesures montrent que les trois familles de modèles restent compatibles avec
un usage interactif dans les conditions de test.

Sur le scénario S3, le Gradient Boosting présente un coût d'inférence et une
latence inférieurs à la Random Forest dans les mesures produites ci-dessus.

La latence et le coût restent faibles en valeur absolue, mais constituent un
critère complémentaire pour l'arbitrage entre modèles.

## 5.5 Optimisation des hyperparamètres

Suite à la comparaison précédente, le scénario S3 est retenu comme périmètre de données principal.

Deux modèles sont sélectionnés pour l'optimisation des hyperparamètres :
- Gradient Boosting, pour maximiser la précision et la ROC-AUC
- Random Forest, pour maximiser le F1-score et le rappel

La Logistic Regression est écartée à ce stade, car elle ne permet pas d'obtenir un compromis satisfaisant entre précision et rappel.

L'objectif de cette étape est d'observer l'effet de quelques hyperparamètres clés sur les performances des modèles, et de déterminer si un gain significatif peut être obtenu par rapport aux modèles par défaut.

In [ ]:
# Paramètres Random Forest
rf_configs = [
    {
        "name": "RF_baseline",
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_leaf": 1,
    },
    {
        "name": "RF_depth_10",
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_leaf": 1,
    },
    {
        "name": "RF_depth_10_leaf_2",
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_leaf": 2,
    },
]

In [ ]:
# Paramètres Gradient Boosting
gb_configs = [
    {
        "name": "GB_baseline",
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 3,
    },
    {
        "name": "GB_slow_learning",
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 3,
    },
    {
        "name": "GB_shallow",
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 2,
    },
]

In [ ]:
features = scenarios["S3_without_duration_or_sensitive"]

num_features, cat_features = get_feature_types(
    X_train,
    features
)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

tuning_results = []

In [ ]:
for config in rf_configs:

    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        random_state=RANDOM_STATE,
    )

    preprocessor = make_preprocessor(
        num_features=num_features,
        cat_features=cat_features,
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    scores = cross_validate(
        pipe,
        X_train[features],
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )

    tuning_results.append({
        "model": "RandomForest",
        "configuration": config["name"],
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
        "roc_auc": scores["test_roc_auc"].mean(),
    })

In [ ]:
for config in gb_configs:

    model = GradientBoostingClassifier(
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        random_state=RANDOM_STATE,
    )

    preprocessor = make_preprocessor(
        num_features=num_features,
        cat_features=cat_features,
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    scores = cross_validate(
        pipe,
        X_train[features],
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )

    tuning_results.append({
        "model": "GradientBoosting",
        "configuration": config["name"],
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
        "roc_auc": scores["test_roc_auc"].mean(),
    })

In [ ]:
tuning_results = pd.DataFrame(tuning_results)

tuning_results.round(3)

### Analyse des configurations

Pour la Random Forest, limiter la profondeur des arbres à 10 améliore fortement
la précision et la ROC-AUC par rapport à la configuration par défaut.

La précision passe de 0.526 à 0.653 et la ROC-AUC de 0.752 à 0.798.
En revanche, le rappel diminue de 0.306 à 0.249, ce qui entraîne également
une baisse du F1-score.

L'ajout de `min_samples_leaf=2` n'apporte pas d'amélioration supplémentaire :
les résultats sont quasiment identiques à ceux obtenus avec `max_depth=10`.

Ces résultats montrent qu'un modèle plus régularisé devient ici plus sélectif :
il identifie moins de souscripteurs potentiels, mais ses prédictions positives
sont plus fiables.

Pour le Gradient Boosting, diminuer le `learning_rate` tout en augmentant le
nombre d'estimateurs modifie très peu les performances.

La configuration avec des arbres moins profonds (`max_depth=2`) augmente la
précision jusqu'à 0.693, mais réduit nettement le rappel à 0.209 et le F1-score
à 0.321.

Le gain de précision s'accompagne donc ici d'une perte importante de couverture
des souscripteurs potentiels.

### Synthèse

Aucune configuration ne domine sur toutes les métriques.

La Random Forest par défaut conserve le meilleur compromis en termes de rappel
et de F1-score.

À l'inverse, les configurations régularisées de Random Forest ainsi que le
Gradient Boosting obtiennent une meilleure précision, ce qui peut être
intéressant dans un contexte où le nombre d'appels disponibles est limité.

Le choix final devra donc tenir compte du compromis métier entre limiter les
appels inutiles et identifier le plus grand nombre possible de souscripteurs.

### 5.5.1 Mesure opérationnelle de la configuration retenue

La configuration finale de la Random Forest étant différente de celle utilisée
lors du benchmark initial, une mesure complémentaire est réalisée afin d'obtenir
les valeurs de latence et de coût CPU correspondant exactement au modèle retenu.

In [ ]:
features = scenarios["S3_without_duration_or_sensitive"]

num_features, cat_features = get_feature_types(
    X_train,
    features
)

preprocessor = make_preprocessor(
    num_features=num_features,
    cat_features=cat_features,
)

final_benchmark_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=RANDOM_STATE,
)

final_benchmark_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_benchmark_model),
])

final_benchmark_pipe.fit(
    X_train[features],
    y_train
)

In [ ]:
X_bench = X_train[features].sample(
    n=min(1000, len(X_train)),
    random_state=RANDOM_STATE
)

# Warm-up
final_benchmark_pipe.predict_proba(
    X_bench.iloc[:10]
)

# -----------------------------
# Latence unitaire
# -----------------------------
latencies = []

for i in range(len(X_bench)):
    row = X_bench.iloc[[i]]

    start = time.perf_counter()

    final_benchmark_pipe.predict_proba(row)

    elapsed = time.perf_counter() - start

    latencies.append(
        elapsed * 1000
    )

# -----------------------------
# Coût CPU / 1000 prédictions
# -----------------------------
repeats = 50

cpu_start = time.process_time()

for _ in range(repeats):
    final_benchmark_pipe.predict_proba(X_bench)

cpu_seconds = time.process_time() - cpu_start

cpu_s_per_1000 = (
    cpu_seconds
    / repeats
    * (1000 / len(X_bench))
)

# -----------------------------
# Résultats
# -----------------------------
final_benchmark_results = {
    "scenario": "S3_without_duration_or_sensitive",
    "model": "RandomForest_200_depth10",
    "latency_mean_ms": np.mean(latencies),
    "latency_p95_ms": np.percentile(latencies, 95),
    "cpu_s_per_1000": cpu_s_per_1000,
}

pd.Series(final_benchmark_results)

### 5.5.2 Arbitrage métier par capacité d'appels

Jusqu'ici, les modèles ont principalement été comparés à partir d'un seuil de
classification fixé à 0.5.

Or ce seuil n'a pas de signification métier particulière. Dans le contexte d'une
campagne téléphonique, la contrainte réelle est plutôt le nombre de clients que la
banque est capable de contacter.

Nous comparons donc la Random Forest retenue et le Gradient Boosting sur plusieurs
capacités de campagne : 5 %, 10 %, 20 % et 30 % des clients.

Pour chaque capacité, les clients sont classés par probabilité décroissante de
souscription et seuls les premiers sont sélectionnés.

Les indicateurs utilisés sont :

- **Précision** : taux réel de souscription parmi les clients sélectionnés ;
- **Souscripteurs captés** : part de tous les souscripteurs effectivement présents
  dans la population sélectionnée ;
- **Lift** : rapport entre le taux de souscription des clients sélectionnés et le
  taux de souscription moyen de la population ;
- **Seuil implicite** : probabilité du dernier client inclus dans la sélection.

Afin de ne pas utiliser le jeu de test pour choisir cette politique, cette analyse
est réalisée uniquement sur le jeu d'entraînement avec des probabilités
*out-of-fold* obtenues par validation croisée.

In [ ]:
features = scenarios["S3_without_duration_or_sensitive"]

num_features, cat_features = get_feature_types(
    X_train,
    features
)

# -----------------------------
# Random Forest retenue
# -----------------------------
rf_topk_pipe = Pipeline([
    (
        "preprocessor",
        make_preprocessor(
            num_features=num_features,
            cat_features=cat_features
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=RANDOM_STATE
        )
    )
])

# -----------------------------
# Gradient Boosting de référence
# -----------------------------
gb_topk_pipe = Pipeline([
    (
        "preprocessor",
        make_preprocessor(
            num_features=num_features,
            cat_features=cat_features
        )
    ),
    (
        "model",
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE
        )
    )
])

# Probabilités out-of-fold :
# chaque observation est prédite par un modèle qui ne l'a pas vue à l'entraînement.
rf_oof_proba = cross_val_predict(
    rf_topk_pipe,
    X_train[features],
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

gb_oof_proba = cross_val_predict(
    gb_topk_pipe,
    X_train[features],
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]


def evaluate_top_k(y_true, probabilities, model_name, capacities):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    baseline_rate = y_true.mean()
    total_positives = y_true.sum()

    rows = []

    for capacity in capacities:
        n_selected = int(np.ceil(len(y_true) * capacity))

        # Indices triés de la probabilité la plus élevée à la plus faible
        selected_idx = np.argsort(probabilities)[::-1][:n_selected]

        y_selected = y_true[selected_idx]
        proba_selected = probabilities[selected_idx]

        true_positives = y_selected.sum()

        precision = y_selected.mean()
        recall = true_positives / total_positives
        lift = precision / baseline_rate
        threshold = proba_selected.min()

        rows.append({
            "model": model_name,
            "capacity": capacity,
            "n_called": n_selected,
            "subscribers_found": int(true_positives),
            "precision": precision,
            "subscribers_captured": recall,
            "lift": lift,
            "implicit_threshold": threshold
        })

    return rows


capacities = [0.05, 0.10, 0.20, 0.30]

topk_results = []

topk_results += evaluate_top_k(
    y_train,
    rf_oof_proba,
    "Random Forest",
    capacities
)

topk_results += evaluate_top_k(
    y_train,
    gb_oof_proba,
    "Gradient Boosting",
    capacities
)

topk_df = pd.DataFrame(topk_results)

display(
    topk_df.style.format({
        "capacity": "{:.0%}",
        "precision": "{:.1%}",
        "subscribers_captured": "{:.1%}",
        "lift": "{:.2f}x",
        "implicit_threshold": "{:.3f}"
    })
)

In [ ]:
rf_operational = pd.Series(
    final_benchmark_results
)

gb_operational = (
    benchmark_results[
        (benchmark_results["scenario"] == "S3_without_duration_or_sensitive")
        & (benchmark_results["model"] == "GradientBoostingClassifier")
    ]
    .iloc[0]
)

operational_comparison = pd.DataFrame([
    {
        "model": "Random Forest",
        "cpu_s_per_1000": rf_operational["cpu_s_per_1000"],
        "latency_p95_ms": rf_operational["latency_p95_ms"],
    },
    {
        "model": "Gradient Boosting",
        "cpu_s_per_1000": gb_operational["cpu_s_per_1000"],
        "latency_p95_ms": gb_operational["latency_p95_ms"],
    },
])

operational_comparison

### Analyse

Cette analyse permet de comparer les modèles dans une situation plus proche de
l'usage réel : la banque dispose d'une capacité d'appels limitée et contacte en
priorité les clients obtenant les scores les plus élevés.

Le choix final de la capacité et du modèle sera effectué à partir :

1. du taux de souscription obtenu parmi les clients appelés ;
2. de la proportion totale de souscripteurs captés ;
3. du lift par rapport à un ciblage non priorisé ;
4. du coût d'inférence ;
5. d'une hypothèse explicite sur la capacité de la campagne.

L'analyse est réalisée avant toute réévaluation sur le jeu de test afin de conserver
celui-ci comme évaluation finale indépendante.

In [ ]:
# Evaluation finale du Gradient Boosting candidat
features = scenarios["S3_without_duration_or_sensitive"]

num_features, cat_features = get_feature_types(
    X_train,
    features
)

gb_final_pipe = Pipeline([
    (
        "preprocessor",
        make_preprocessor(
            num_features=num_features,
            cat_features=cat_features
        )
    ),
    (
        "model",
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE
        )
    )
])

# Entraînement uniquement sur le train
gb_final_pipe.fit(
    X_train[features],
    y_train
)

# Probabilités sur le test
gb_test_proba = gb_final_pipe.predict_proba(
    X_test[features]
)[:, 1]

# Export des résultats de test pour le dashboard de monitoring
campaign_results = pd.DataFrame({
    "probability": gb_test_proba,
    "y_true": np.asarray(y_test),
})

monitoring_dir = Path("../monitoring")
monitoring_dir.mkdir(parents=True, exist_ok=True)

campaign_results.to_csv(
    monitoring_dir / "campaign_results.csv",
    index=False,
)

# Politique métier choisie : top 10 %
n_top10 = int(np.ceil(len(y_test) * 0.10))

top10_idx = np.argsort(gb_test_proba)[::-1][:n_top10]

y_top10 = np.asarray(y_test)[top10_idx]
proba_top10 = gb_test_proba[top10_idx]

baseline_test = y_test.mean()

gb_test_top10 = {
    "population_test": len(y_test),
    "n_called": n_top10,
    "capacity": 0.10,
    "subscribers_found": int(y_top10.sum()),
    "precision": y_top10.mean(),
    "subscribers_captured": y_top10.sum() / y_test.sum(),
    "lift": y_top10.mean() / baseline_test,
    "implicit_threshold": proba_top10.min()
}

pd.Series(gb_test_top10)

In [ ]:
gb_test_pred_05 = (gb_test_proba >= 0.5).astype(int)

gb_test_metrics_05 = {
    "precision": precision_score(y_test, gb_test_pred_05),
    "recall": recall_score(y_test, gb_test_pred_05),
    "f1": f1_score(y_test, gb_test_pred_05),
    "roc_auc": roc_auc_score(y_test, gb_test_proba)
}

print(pd.Series(gb_test_metrics_05))
print()
print(confusion_matrix(y_test, gb_test_pred_05))

### 5.5.3 Choix du modèle et de la politique de ciblage

Les résultats montrent que Random Forest et Gradient Boosting classent les
clients de manière très similaire lorsque l'on raisonne à capacité d'appels
constante.

Une hypothèse de travail est retenue pour la suite : la campagne peut contacter
environ 10 % de la population cible.

Sur les prédictions out-of-fold du jeu d'entraînement :

- Random Forest atteint une précision de 50,7 %, capture 45,0 % des souscripteurs
  et obtient un lift de 4,50 ;
- Gradient Boosting atteint une précision de 50,9 %, capture 45,2 % des
  souscripteurs et obtient un lift de 4,52.

Les performances métier sont donc quasiment équivalentes.

Gradient Boosting présente cependant un coût d'inférence inférieur et une latence
plus faible. Il est donc retenu comme modèle final.

La politique de ciblage retenue consiste à classer les clients par probabilité
prédite de souscription et à sélectionner les 10 % ayant les scores les plus
élevés.

Le seuil observé pour atteindre cette capacité sur les prédictions out-of-fold
est d'environ 0,319. Ce seuil n'est toutefois pas figé : dans une campagne
réelle, la sélection repose prioritairement sur le classement des scores et sur
la capacité d'appels disponible.

## 5.6 Evaluation finale sur le jeu de test

Le scénario S3, le modèle Gradient Boosting et la politique de ciblage à 10 %
ont été retenus à partir des résultats obtenus uniquement sur le jeu
d'entraînement.

Le jeu de test n'a pas été utilisé pour choisir le modèle, ses hyperparamètres
ou la capacité de campagne. Il est utilisé ici une seule fois afin d'estimer
les performances finales de la solution.

Le modèle final est un `GradientBoostingClassifier` utilisant :

- `n_estimators=100`
- `learning_rate=0.1`
- `max_depth=3`

Il est évalué selon deux lectures complémentaires :

1. les métriques classiques au seuil 0.5, afin de rester comparable aux modèles
   étudiés précédemment
2. la politique métier retenue, consistant à contacter les 10 % de clients ayant
   les scores les plus élevés

Ce choix est cohérent avec l'objectif métier de limiter les sollicitations
inutiles, sans privilégier exclusivement la précision au détriment de la
détection des souscripteurs potentiels.

In [ ]:
features = scenarios["S3_without_duration_or_sensitive"]

num_features, cat_features = get_feature_types(
    X_train,
    features
)

preprocessor = make_preprocessor(
    num_features=num_features,
    cat_features=cat_features
)

final_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=RANDOM_STATE
)

final_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model)
])

final_pipe.fit(
    X_train[features],
    y_train
)

y_proba = final_pipe.predict_proba(
    X_test[features]
)[:, 1]

y_pred_05 = (y_proba >= 0.5).astype(int)

In [ ]:
test_results_05 = {
    "precision": precision_score(y_test, y_pred_05),
    "recall": recall_score(y_test, y_pred_05),
    "f1": f1_score(y_test, y_pred_05),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

pd.Series(test_results_05).round(3)

In [ ]:
print("Métriques au seuil 0.5")
display(pd.Series(test_results_05).round(3))

print("\nMatrice de confusion au seuil 0.5")
print(confusion_matrix(y_test, y_pred_05))

print("\nRapport de classification au seuil 0.5")
print(classification_report(y_test, y_pred_05, digits=3))

In [ ]:
n_top10 = int(np.ceil(len(y_test) * 0.10))

top10_idx = np.argsort(y_proba)[::-1][:n_top10]

y_top10 = np.asarray(y_test)[top10_idx]
proba_top10 = y_proba[top10_idx]

top10_results = {
    "population_test": len(y_test),
    "clients_appeles": n_top10,
    "precision": y_top10.mean(),
    "souscripteurs_trouves": int(y_top10.sum()),
    "souscripteurs_captes": y_top10.sum() / y_test.sum(),
    "lift": y_top10.mean() / y_test.mean(),
    "seuil_observe": proba_top10.min()
}

pd.Series(top10_results)

### Lecture des résultats

Au seuil classique de 0.5, le modèle est très sélectif : sa précision est élevée,
mais son rappel reste faible. Il identifie environ un quart des souscripteurs réels.

Cette lecture est utile pour comparer le modèle aux expériences précédentes, mais
elle ne correspond pas directement à la politique métier retenue.

Avec une capacité d'appels limitée à 10 % de la population, les clients sont
classés par score décroissant et seuls les 10 % ayant les probabilités les plus
élevées sont sélectionnés.

Sur le jeu de test, cette politique conduit à :

- 824 clients ciblés sur 8 238
- 453 souscripteurs identifiés
- 55,0 % de souscription parmi les clients ciblés
- 48,8 % de l'ensemble des souscripteurs captés
- un lift de 4,88 par rapport au taux moyen de souscription

Le seuil observé sur ce jeu est d'environ 0,325, mais il n'est pas utilisé comme
seuil fixe en production : la politique repose sur le classement des scores et
sur la capacité réelle de la campagne.

## 5.7 Synthèse Modélisation

Le modèle final retenu est un Gradient Boosting appliqué au scénario S3.

Le scénario S3 exclut `duration`, indisponible avant l'appel, ainsi que les
variables socio-démographiques `age`, `job`, `marital` et `education`.

Le choix final ne repose pas uniquement sur un seuil de classification fixé à
0.5. Le besoin métier est reformulé en contrainte de capacité : la campagne est
supposée pouvoir contacter 10 % de la population.

Sur le jeu de test, la politique top 10 % conduit aux résultats suivants :

- 824 clients contactés sur 8 238
- 453 souscripteurs identifiés
- taux de souscription parmi les clients ciblés : 55,0 %
- 48,8 % de l'ensemble des souscripteurs captés
- lift : 4,88 par rapport au taux moyen de souscription

À titre de comparaison, au seuil classique de 0.5, le modèle obtient :

- précision : 0.686
- rappel : 0.247
- F1-score : 0.363
- ROC-AUC : 0.810

Le seuil de 0.5 produit donc un modèle très sélectif mais laisse de côté une
grande partie des souscripteurs potentiels.

La politique basée sur le classement des probabilités et sur la capacité de
campagne est plus directement alignée avec l'usage métier.

### Persistance du modèle final

In [ ]:
joblib.dump(
    final_pipe,
    "../models/pipeline.joblib"
)

## 6. Analyse des scénarios & arbitrages

### 6.1 Tableau comparatif

Les quatre scénarios sont comparés selon leurs performances prédictives, leur
compatibilité avec un ciblage avant appel, leur coût opérationnel et les risques
éthiques associés.

In [ ]:
representative_models = pd.DataFrame([
    {
        "scenario": "S1_all",
        "model": "GradientBoostingClassifier",
        "Scénario": "S1 — Toutes variables",
        "Risques / limites": (
            "`duration` indisponible avant l'appel "
            "+ variables socio-démographiques"
        ),
        "Verdict": "Référence de performance uniquement",
    },
    {
        "scenario": "S2_without_duration",
        "model": "RandomForestClassifier",
        "Scénario": "S2 — Sans duration",
        "Risques / limites": (
            "Variables socio-démographiques conservées"
        ),
        "Verdict": "Exploitable mais moins favorable",
    },
    {
        "scenario": "S3_without_duration_or_sensitive",
        "model": "GradientBoostingClassifier",
        "Scénario": (
            "S3 — Sans duration ni variables socio-démographiques"
        ),
        "Risques / limites": (
            "Des biais indirects peuvent subsister"
        ),
        "Verdict": "Scénario et modèle retenus",
    },
    {
        "scenario": "S4_minimal",
        "model": "RandomForestClassifier",
        "Scénario": "S4 — Minimal",
        "Risques / limites": (
            "Réduction supplémentaire des données "
            "mais perte de performance"
        ),
        "Verdict": "Plus sobre mais moins performant",
    },
])


comparison_6_1 = (
    representative_models
    .merge(
        cv_results[
            [
                "scenario",
                "model",
                "f1_mean",
                "precision_mean",
                "roc_auc_mean",
            ]
        ],
        on=["scenario", "model"],
        how="left",
    )
    .merge(
        benchmark_results[
            [
                "scenario",
                "model",
                "cpu_s_per_1000",
                "latency_p95_ms",
            ]
        ],
        on=["scenario", "model"],
        how="left",
    )
)


comparison_6_1["Modèle représentatif"] = (
    comparison_6_1["model"]
    .replace({
        "GradientBoostingClassifier": "Gradient Boosting",
        "RandomForestClassifier": "Random Forest",
        "LogisticRegression": "Logistic Regression",
    })
)


comparison_6_1 = comparison_6_1[
    [
        "Scénario",
        "Modèle représentatif",
        "f1_mean",
        "precision_mean",
        "roc_auc_mean",
        "cpu_s_per_1000",
        "latency_p95_ms",
        "Risques / limites",
        "Verdict",
    ]
].rename(columns={
    "f1_mean": "F1 CV",
    "precision_mean": "Précision CV",
    "roc_auc_mean": "ROC-AUC CV",
    "cpu_s_per_1000": "CPU·s / 1000 préd.",
    "latency_p95_ms": "Latence p95 (ms)",
})


comparison_6_1.style.format({
    "F1 CV": "{:.3f}",
    "Précision CV": "{:.3f}",
    "ROC-AUC CV": "{:.3f}",
    "CPU·s / 1000 préd.": "{:.3f}",
    "Latence p95 (ms)": "{:.2f}",
})

Le scénario S3 reste le meilleur compromis entre faisabilité pré-appel,
performances et réduction de l'utilisation directe des variables
socio-démographiques.

Au sein de S3, Random Forest et Gradient Boosting produisent des performances
métier très proches lorsque l'on raisonne à capacité d'appels fixe.

Gradient Boosting est finalement retenu car il présente un léger avantage sur
les métriques métier top-k ainsi qu'un coût d'inférence inférieur dans les
mesures réalisées.

### 6.2 Recommandation finale au client

Nous recommandons le scénario S3 associé à un modèle Gradient Boosting.

Ce scénario permet d'effectuer le ciblage avant l'appel sans utiliser directement
`age`, `job`, `marital`, `education` ni la variable `duration`, qui n'est connue
qu'après l'appel.

Plutôt que d'utiliser le seuil de classification standard de 0.5, la stratégie
proposée consiste à classer les clients selon leur probabilité estimée de
souscription puis à adapter le nombre de clients contactés à la capacité réelle
de la campagne.

Avec l'hypothèse de travail d'une capacité limitée à 10 % de la population, le
modèle permet sur le jeu de test de concentrer les appels sur une population dont
le taux de souscription atteint environ 55 %, contre environ 11 % dans la
population globale sans ciblage par le modèle.

Cette stratégie correspond à un lift d'environ 4,9 et permet de capter près de
49 % de l'ensemble des souscripteurs.

Cette politique permet donc de réduire fortement le volume de sollicitations tout
en conservant une part importante des opportunités commerciales.

### 6.3 📝 Synthèse arbitrages

Le scénario S1 obtient les meilleures performances prédictives, mais ne peut pas
être utilisé pour un ciblage avant appel en raison de la présence de `duration`.

Le scénario S2 supprime cette fuite de données, mais conserve les variables
socio-démographiques.

Le scénario S3 retire également `age`, `job`, `marital` et `education` sans
entraîner de dégradation importante des performances par rapport à S2.

Random Forest et Gradient Boosting ont ensuite été comparés plus précisément sur
ce scénario. Une analyse à capacité d'appels constante montre que leurs
performances métier sont très proches.

Pour une capacité de campagne fixée à 10 %, Gradient Boosting obtient légèrement
de meilleurs résultats de ciblage tout en présentant un coût d'inférence et une
latence inférieurs.

Le choix final repose donc sur :

- le scénario S3
- un modèle Gradient Boosting
- une politique de classement des clients par score
- une capacité d'appels utilisée comme contrainte métier plutôt qu'un seuil fixe
  de classification

## 7. Interprétation pour la communication client

### 7.1 Importance des variables

Afin de rendre le fonctionnement du modèle plus compréhensible, nous évaluons
l'importance des variables utilisées par le modèle final.

L'importance par permutation consiste à mélanger aléatoirement les valeurs
d'une variable et à mesurer la dégradation des performances du modèle.
Plus la performance diminue, plus le modèle dépend de cette variable.

Cette analyse permet d'identifier les variables les plus utiles à la prédiction,
sans pour autant établir de relation causale.

In [ ]:
features = scenarios["S3_without_duration_or_sensitive"]

permutation_results = permutation_importance(
    final_pipe,
    X_test[features],
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [ ]:
importance_df = pd.DataFrame({
    "feature": features,
    "importance_mean": permutation_results.importances_mean,
    "importance_std": permutation_results.importances_std,
})

importance_df = importance_df.sort_values(
    by="importance_mean",
    ascending=False
)

importance_df

In [ ]:
top_features = importance_df.head(10).sort_values(
    by="importance_mean"
)

plt.figure(figsize=(8, 5))

plt.barh(
    top_features["feature"],
    top_features["importance_mean"],
    xerr=top_features["importance_std"]
)

plt.xlabel("Baisse moyenne de ROC-AUC après permutation")
plt.ylabel("Variable")
plt.title("Importance des variables du modèle final")

plt.show()

L'analyse par permutation montre que le Gradient Boosting final s'appuie
principalement sur des variables liées au contexte économique et aux conditions
de contact.

La variable `nr.employed` est de loin la plus importante : sa permutation
entraîne la plus forte diminution de ROC-AUC. Elle est suivie par `euribor3m`
et `cons.conf.idx`.

Les variables `month` et `contact` jouent également un rôle notable, ce qui
indique que le moment et le canal de contact participent au classement produit
par le modèle.

L'historique de campagne contribue également à la prédiction via `pdays` et
`poutcome`, mais avec une importance plus faible.

À l'inverse, `housing` et `loan` présentent une importance par permutation
proche de zéro. Leur présence apporte donc peu d'information supplémentaire
au modèle dans cette configuration.

Ces importances mesurent la dépendance prédictive du modèle aux différentes
variables. Elles ne doivent pas être interprétées comme des relations causales.

### 7.2 Analyse des erreurs et stratégies de fallback

L'objectif est d'analyser les situations dans lesquelles le modèle produit des
erreurs et de vérifier si son comportement diffère selon certains groupes de
population.

Deux lectures sont distinguées :

- une lecture de classification classique au seuil de 0.5, utilisée pour analyser
  les faux positifs et faux négatifs
- une lecture métier correspondant à la politique de ciblage top 10 %, utilisée
  pour analyser quels clients sont effectivement sélectionnés pour la campagne

L'analyse porte sur :

- la nature des faux positifs et faux négatifs
- les probabilités associées aux erreurs
- les performances par sous-groupe
- le taux de sélection par sous-groupe
- la concentration éventuelle des erreurs ou des sélections sur certains profils

In [ ]:
# Prédiction classique au seuil 0.5
y_pred = (y_proba >= 0.5).astype(int)

error_df = X_test.copy()

error_df["y_true"] = np.asarray(y_test)
error_df["y_pred"] = y_pred
error_df["proba_yes"] = y_proba

error_df["error_type"] = "correct"

error_df.loc[
    (error_df["y_true"] == 1) & (error_df["y_pred"] == 0),
    "error_type"
] = "FN"

error_df.loc[
    (error_df["y_true"] == 0) & (error_df["y_pred"] == 1),
    "error_type"
] = "FP"

error_df["error_type"].value_counts()

In [ ]:
fp = error_df[error_df["error_type"] == "FP"]
fn = error_df[error_df["error_type"] == "FN"]

error_probability_summary = pd.DataFrame({
    "type": ["FP", "FN"],
    "count": [len(fp), len(fn)],
    "mean_probability": [
        fp["proba_yes"].mean(),
        fn["proba_yes"].mean()
    ],
    "median_probability": [
        fp["proba_yes"].median(),
        fn["proba_yes"].median()
    ],
    "min_probability": [
        fp["proba_yes"].min(),
        fn["proba_yes"].min()
    ],
    "max_probability": [
        fp["proba_yes"].max(),
        fn["proba_yes"].max()
    ]
})

error_probability_summary

Les faux positifs ont une probabilité moyenne de souscription d'environ 0,63,
tandis que les faux négatifs ont une probabilité moyenne d'environ 0,20 et une
médiane d'environ 0,14.

Les faux négatifs ne sont donc pas uniquement concentrés à proximité du seuil
de 0.5 : une partie importante des souscripteurs non détectés reçoit un score
nettement inférieur.

Une simple zone d'incertitude autour de 0.5 ne permettrait donc pas de corriger
le principal défaut du modèle.

In [ ]:
error_df["age_group"] = pd.cut(
    error_df["age"],
    bins=[0, 29, 39, 49, 59, np.inf],
    labels=["<30", "30-39", "40-49", "50-59", "60+"]
)

In [ ]:
def subgroup_error_metrics(group):
    y_true_group = group["y_true"]
    y_pred_group = group["y_pred"]

    positives = (y_true_group == 1).sum()
    negatives = (y_true_group == 0).sum()

    fn = ((y_true_group == 1) & (y_pred_group == 0)).sum()
    fp = ((y_true_group == 0) & (y_pred_group == 1)).sum()

    fnr = fn / positives if positives > 0 else np.nan
    fpr = fp / negatives if negatives > 0 else np.nan

    return pd.Series({
        "count": len(group),
        "FNR": fnr,
        "FPR": fpr
    })


error_by_age = (
    error_df
    .groupby("age_group", observed=False)
    .apply(subgroup_error_metrics, include_groups=False)
)

error_by_age.style.format({
    "FNR": "{:.1%}",
    "FPR": "{:.1%}"
})

In [ ]:
error_by_job = (
    error_df
    .groupby("job")
    .apply(subgroup_error_metrics, include_groups=False)
    .sort_values("FNR")
)

error_by_job.style.format({
    "FNR": "{:.1%}",
    "FPR": "{:.1%}"
})

Les taux d'erreur diffèrent sensiblement selon les sous-groupes, alors même que
`age` et `job` ne sont pas utilisés comme variables d'entrée du modèle final.

Par exemple, le FNR varie de 57,3 % chez les clients de 60 ans et plus à 80,3 %
chez les 30-39 ans. Des écarts importants apparaissent également selon la
profession : le FNR atteint notamment 91,1 % pour les `blue-collar`, contre
58,1 % pour les retraités.

Les taux de faux positifs varient également, avec notamment un FPR plus élevé
chez les 60 ans et plus.

Ces écarts ne permettent pas à eux seuls de conclure à une discrimination :
les taux de souscription réels et la représentation des groupes diffèrent
également. Ils montrent néanmoins que le retrait direct des variables
socio-démographiques ne suffit pas à garantir un comportement homogène du
modèle entre les groupes.

In [ ]:
# Politique métier :
# sélectionner les 10 % de clients ayant les probabilités les plus élevées

n_top10 = int(np.ceil(len(error_df) * 0.10))

top10_idx = np.argsort(error_df["proba_yes"].to_numpy())[::-1][:n_top10]

selected_top10 = np.zeros(len(error_df), dtype=int)
selected_top10[top10_idx] = 1

error_df["selected_top10"] = selected_top10

error_df["selected_top10"].value_counts()

In [ ]:
selection_by_age = (
    error_df
    .groupby("age_group", observed=False)["selected_top10"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "selection_rate"})
)

selection_by_age["selection_ratio_vs_max"] = (
    selection_by_age["selection_rate"]
    / selection_by_age["selection_rate"].max()
)

selection_by_age.style.format({
    "selection_rate": "{:.1%}",
    "selection_ratio_vs_max": "{:.3f}"
})

In [ ]:
age_selection_vs_subscription = (
    selection_by_age[
        ["count", "selection_rate"]
    ]
    .copy()
)

age_selection_vs_subscription["subscription_rate"] = (
    age_subscription_rate
    .reindex(age_selection_vs_subscription.index)
    / 100
)

age_selection_vs_subscription["selection_gap"] = (
    age_selection_vs_subscription["selection_rate"]
    - age_selection_vs_subscription["subscription_rate"]
)

age_selection_vs_subscription.style.format({
    "selection_rate": "{:.1%}",
    "subscription_rate": "{:.1%}",
    "selection_gap": "{:+.1%}",
})

### Comparaison entre sélection et souscription réelle

La comparaison entre le taux de sélection et le taux réel de souscription permet
de distinguer les différences présentes dans les données des effets de la
politique de ciblage.

Pour les tranches de moins de 60 ans, les écarts restent relativement limités :
la sélection est proche ou légèrement inférieure au taux réel de souscription.

En revanche, les clients de 60 ans et plus présentent un taux réel de
souscription de 45,5 %, mais un taux de sélection de 54,5 %, soit un écart de
+9 points.

La forte présence de ce groupe dans le top 10 % s'explique donc en partie par
un taux de souscription historiquement plus élevé, mais la politique de ciblage
accentue également cette concentration.

Ces résultats ne suffisent pas à conclure à une discrimination du modèle, mais
ils justifient une vigilance particulière dans le cadre d'un produit d'épargne
commercialisé par téléphone.

La recommandation est donc de conserver `age` hors des variables de prédiction,
de suivre les taux de sélection et d'erreur par tranche d'âge et de prévoir, au
niveau du processus de campagne, des règles limitant la fréquence des
sollicitations indépendamment du score.

In [ ]:
selection_by_job = (
    error_df
    .groupby("job")["selected_top10"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "selection_rate"})
)

selection_by_job["selection_ratio_vs_max"] = (
    selection_by_job["selection_rate"]
    / selection_by_job["selection_rate"].max()
)

selection_by_job = selection_by_job.sort_values(
    "selection_rate",
    ascending=False
)

selection_by_job.style.format({
    "selection_rate": "{:.1%}",
    "selection_ratio_vs_max": "{:.3f}"
})

La politique top 10 % produit également des taux de sélection très différents
selon les groupes.

Par tranche d'âge, 54,5 % des clients de 60 ans et plus appartiennent au top
10 % des scores, contre 6,3 % des 40-49 ans.

Les écarts sont également importants selon la profession : 41,5 % des étudiants
et 32,2 % des retraités sont sélectionnés, contre seulement 2,6 % des
`blue-collar` et 5,2 % des clients travaillant dans les services.

Ces différences sont observées alors que `age` et `job` ne sont pas fournis au
modèle. Elles peuvent provenir de variables restantes corrélées à ces
caractéristiques ainsi que de différences réelles de taux de souscription entre
les groupes.

Le ratio présenté ici compare chaque taux de sélection au taux maximal observé.
Il constitue un indicateur exploratoire d'écart de sélection et ne doit pas être
interprété isolément comme une preuve de discrimination.

In [ ]:
zone = error_df[
    error_df["proba_yes"].between(0.40, 0.60)
]

zone_summary = pd.Series({
    "n_clients": len(zone),
    "part_population": len(zone) / len(error_df),
    "subscription_rate": zone["y_true"].mean(),
    "baseline_rate": error_df["y_true"].mean(),
    "lift_vs_baseline": (
        zone["y_true"].mean()
        / error_df["y_true"].mean()
    )
})

zone_summary

### Analyse de l'ancienne zone d'incertitude

La zone de probabilité `[0.40 ; 0.60]` contient 279 clients, soit environ 3,4 %
du jeu de test.

Le taux réel de souscription dans cette population atteint 53,0 %, contre 11,3 %
sur l'ensemble du jeu de test, soit un lift d'environ 4,71.

Cette population n'est donc pas une population peu intéressante ou simplement
"incertaine" : elle présente au contraire une probabilité réelle de souscription
très supérieure à la moyenne.

La stratégie initialement envisagée consistant à s'abstenir automatiquement
entre 0.40 et 0.60 n'est donc pas retenue.

La politique finale repose plutôt sur le classement des probabilités et sur la
capacité disponible pour la campagne. Pour une capacité de 10 %, les clients
ayant les scores les plus élevés sont priorisés, quelle que soit leur proximité
avec le seuil classique de 0.5.

### 7.3 Message au client

Le modèle final est utilisé comme un outil de priorisation de campagne.

Il attribue à chaque client une probabilité estimée de souscription, puis les
clients sont classés par score décroissant.

La politique opérationnelle étudiée consiste à sélectionner les 10 % de clients
ayant les scores les plus élevés, en cohérence avec une hypothèse de capacité
limitée du centre d'appels.

Sur le jeu de test, cette stratégie permet de cibler 824 clients sur 8 238,
d'identifier 453 souscripteurs et d'atteindre un taux de souscription d'environ
55 % parmi les clients sélectionnés, contre environ 11 % dans la population
globale.

Elle permet ainsi de concentrer fortement les appels sur les profils les plus
prometteurs tout en captant près de 49 % de l'ensemble des souscripteurs.

Le modèle ne doit cependant pas être considéré comme une décision automatique
individuelle. Des écarts de taux d'erreur et de sélection persistent entre
certains sous-groupes, malgré le retrait des variables socio-démographiques
directes.

Ces écarts doivent faire l'objet d'un suivi en production.

### 7.4 📝 Synthèse interprétation

Le Gradient Boosting final s'appuie principalement sur des variables liées au
contexte économique, aux conditions de contact et à l'historique des campagnes.

L'analyse au seuil classique de 0.5 montre que les faux négatifs ne sont pas
uniquement concentrés près du seuil : certains souscripteurs reçoivent des scores
nettement plus faibles.

La politique métier finale ne repose donc pas sur ce seuil mais sur le classement
des scores et la capacité disponible pour la campagne.

Pour une capacité de 10 %, le modèle concentre les appels sur une population
présentant un taux de souscription très supérieur à la moyenne.

L'analyse par sous-groupe montre toutefois des écarts importants de taux d'erreur
et de sélection selon l'âge et la profession, même si ces variables ne sont pas
utilisées directement par le modèle.

Le retrait des variables socio-démographiques réduit leur utilisation directe,
mais ne garantit donc pas l'absence de biais indirect.

# 8 Industrialisation

### 8.1 Persistance du modèle

Le modèle final retenu correspond au scénario S3 associé à un
`GradientBoostingClassifier`.

Le pipeline complet est persisté au format `joblib`. Il contient à la fois les
transformations de preprocessing et le modèle afin de garantir que les mêmes
traitements sont appliqués lors de l'entraînement et de l'inférence.

L'artefact utilisé par le service est stocké dans :

`models/pipeline.joblib`

Le pipeline persisté utilise les hyperparamètres suivants :

- `n_estimators=100`
- `learning_rate=0.1`
- `max_depth=3`

La traçabilité repose actuellement sur le dépôt Git, les métriques documentées
dans le notebook et l'artefact du modèle associé au projet.

L'utilisation de MLflow a été envisagée pour assurer un suivi plus structuré des
expériences, des hyperparamètres, des métriques et des différentes versions du
modèle, mais cette brique n'a pas été intégrée dans la version actuelle.

**Artefact servi :** `models/pipeline.joblib`
**Versionnement :** Git
**Suivi d'expériences :** non industrialisé à ce stade

### 8.2 API de service

Repo associé : `git@github.com:anascimento-devid/cas_usage.git` — dossier `api/`

Le modèle est exposé via une API FastAPI.

La version opérationnelle du service repose principalement sur deux routes :

| Route | Méthode | Rôle | Validation |
|---|---|---|---|
| `/health` | GET | Vérifie la disponibilité du service | — |
| `/predict` | POST | Retourne le score individuel de souscription | Pydantic |

Les entrées de `/predict` sont validées avec Pydantic avant d'être transmises
au pipeline de prédiction.

L'API retourne la probabilité estimée de souscription pour le client transmis.

La règle métier finale ne repose pas sur un seuil fixe appliqué à chaque
prédiction. Pour une campagne, les scores de l'ensemble des clients sont classés
par ordre décroissant puis les clients sont sélectionnés selon la capacité
disponible.

Dans l'hypothèse étudiée dans ce projet, cette capacité est fixée à 10 % de la
population.

Une route `/train` existe actuellement uniquement comme placeholder et ne
déclenche pas de processus réel de réentraînement. Le réentraînement est traité
comme une évolution contrôlée du cycle de vie du modèle et non comme une
fonctionnalité disponible de l'API actuelle.

### Preuve de fonctionnement de l'API

L'API FastAPI expose les routes de santé et de scoring décrites précédemment

<p align="center">
  <img src="../docs/swagger.png" width="900">
</p>

### 8.3 Interface utilisateur

Une interface Streamlit permet à un utilisateur non technique de saisir les
informations nécessaires et d'interroger l'API FastAPI.

L'interface affiche le score individuel estimé de souscription retourné par
le modèle.

Cette interface permet principalement de tester le fonctionnement du service sur
un profil donné. Elle ne représente pas à elle seule le processus complet de
sélection d'une campagne.

En effet, la politique métier finale repose sur un classement collectif des
scores : pour sélectionner les 10 % de clients prioritaires, il est nécessaire
de scorer l'ensemble de la population de campagne puis de classer les résultats.

Le frontend ne charge pas directement le modèle : les prédictions passent par
l'API, ce qui permet de centraliser la validation des entrées et l'inférence.

**Code :** `ui/app.py`

### Interface de démonstration

L'interface Streamlit permet à un utilisateur non technique d'interroger le service de scoring

<p align="center">
  <img src="../docs/UI.png" width="900">
</p>

### 8.4 Schéma d'architecture

L'architecture sépare la phase d'entraînement du service d'inférence.

Le pipeline complet de preprocessing et de classification est persisté sous
forme d'un artefact `pipeline.joblib`.

L'API FastAPI charge cet artefact et centralise :

- la validation des données d'entrée ;
- l'application du preprocessing ;
- le calcul de la probabilité de souscription.

L'interface Streamlit communique uniquement avec l'API et n'accède pas
directement au modèle.

La sélection opérationnelle d'une campagne constitue une étape distincte :
l'ensemble des clients éligibles est scoré, puis les scores sont triés afin de
sélectionner la proportion correspondant à la capacité disponible.

```mermaid
graph LR
    D[(Données historiques)]
    T[Entraînement / Validation]
    P[Pipeline preprocessing + Gradient Boosting]
    M[(pipeline.joblib)]

    C[(Clients à scorer)]
    A[API FastAPI]
    S[Scores de souscription]
    B[Classement / capacité campagne]
    U[Interface Streamlit]

    D --> T
    T --> P
    P --> M

    M --> A
    C --> A
    A --> S
    S --> B

    U --> A
```

### 8.5 Conteneurisation & CI/CD

L'API et l'interface utilisateur sont conteneurisées avec Docker afin de garantir
un environnement d'exécution reproductible.

Deux services sont définis dans `docker-compose.yml` :

- `api` : service FastAPI chargé de la validation et du scoring ;
- `ui` : interface Streamlit communiquant avec l'API sur le réseau Docker.

Une pipeline d'intégration continue GitHub Actions est également mise en place.

À chaque `push` ou `pull request`, elle :

1. installe les dépendances ;
2. exécute les tests automatisés avec `pytest` ;
3. vérifie que l'image Docker de l'API peut être construite.

**Fichiers associés :**

- `Dockerfile`
- `Dockerfile.ui`
- `docker-compose.yml`
- `.github/workflows/ci.yml`

Le workflow ne réalise pas actuellement de déploiement automatique vers un
environnement de production ou un registre d'images.

### Tests automatisés

Les principaux comportements de l'API sont vérifiés automatiquement avec Pytest

<p align="center">
  <img src="../docs/tests_ok.png" width="900">
</p>

## 8.6 Synthèse de l'industrialisation

La solution repose sur une architecture simple et découplée adaptée à un cas
d'usage de classification tabulaire.

Le pipeline complet de preprocessing et de Gradient Boosting est persisté dans
un artefact `pipeline.joblib`. Celui-ci est chargé par une API FastAPI qui
centralise la validation des données et le calcul du score individuel de
souscription.

Une interface Streamlit permet d'interroger le service sans manipuler directement
l'API.

La décision de ciblage n'est pas prise individuellement par l'API : les scores
des clients d'une campagne doivent être classés afin de sélectionner la proportion
correspondant à la capacité disponible.

La solution est conteneurisée avec Docker. L'API et l'interface sont exécutées
comme deux services distincts via Docker Compose.

Des tests automatisés avec `pytest` vérifient les principaux comportements du
service, notamment :

- la disponibilité via `/health`
- le fonctionnement de `/predict`
- la validation des données d'entrée
- le rejet des requêtes invalides

Une pipeline CI GitHub Actions exécute automatiquement les tests puis vérifie
la construction de l'image Docker de l'API à chaque `push` ou `pull request`.

Le projet constitue donc un prototype industrialisé reproductible.

Le logging applicatif de l'API et un prototype de dashboard de monitoring
Streamlit sont implémentés.

Le monitoring technique peut être alimenté directement à partir des appels API,
tandis que les métriques métier nécessitent la récupération différée des
résultats réels de campagne.

Pour un déploiement de production complet, il resterait notamment à mettre en
place une supervision persistante du service, une politique de conservation des
données de monitoring et un processus contrôlé de remplacement du modèle.

| Élément | Technologie | Rôle |
|---|---|---|
| Modèle | Gradient Boosting / Scikit-learn | Scoring de probabilité de souscription |
| Préprocessing | Scikit-learn Pipeline | Transformation des données |
| Persistance | Joblib | Stockage du pipeline entraîné |
| API | FastAPI | Exposition du scoring |
| Validation | Pydantic | Validation des entrées |
| Interface | Streamlit | Démonstration du scoring individuel |
| Monitoring | Streamlit | Suivi des métriques techniques et métier |
| Tests | Pytest | Validation automatique des comportements |
| Conteneurisation | Docker | Reproductibilité de l'environnement |
| Orchestration | Docker Compose | Lancement API + interface |
| CI | GitHub Actions | Tests et build Docker automatiques |

### Intégration continue

La pipeline GitHub Actions exécute automatiquement les tests et vérifie la construction de l'image Docker à chaque push ou pull request

<p align="center">
  <img src="../docs/tests_ok_2.png" width="900">
</p>

## 9. Suivi en production & amélioration continue

Le déploiement du modèle ne constitue pas la fin de son cycle de vie.

Les données d'entrée, les comportements de souscription et les conditions de
campagne peuvent évoluer dans le temps. Le suivi doit donc couvrir plusieurs
dimensions complémentaires :

- la disponibilité et les performances techniques du service
- la stabilité des données d'entrée
- la qualité prédictive du modèle lorsque les résultats réels deviennent disponibles
- l'efficacité métier de la politique de ciblage
- les écarts de comportement entre sous-groupes
- la nécessité éventuelle d'un réentraînement

La politique métier finale reposant sur un classement des scores, le monitoring
ne se limite pas aux métriques classiques de classification. Il doit également
suivre les performances obtenues à capacité de campagne constante.

### 9.1 Métriques à surveiller

La surveillance combine des indicateurs techniques, statistiques, prédictifs,
métier et d'équité.

| Type | Métrique | Référence / seuil d'alerte proposé | Source |
|---|---|---|---|
| Système | Latence p95 | alerte si > 500 ms | logs API |
| Système | Taux d'erreur API | alerte si > 1 % | logs API |
| Système | Disponibilité | alerte si < 99 % | supervision du service |
| Données | PSI des variables principales | alerte si PSI > 0.20 | données d'inférence |
| Données | Catégories invalides / inconnues | augmentation inhabituelle | validation API |
| Modèle | ROC-AUC | baisse > 0.05 par rapport à la référence | scores + résultats réels |
| Modèle | Précision au seuil 0.5 | baisse > 0.05 | scores + résultats réels |
| Modèle | Rappel au seuil 0.5 | baisse > 0.05 | scores + résultats réels |
| Modèle | F1-score au seuil 0.5 | baisse > 0.05 | scores + résultats réels |
| Métier | Capacité de campagne | 10 % dans l'hypothèse étudiée | configuration campagne |
| Métier | Précision top 10 % | référence ≈ 55,0 % sur le jeu de test | scores + résultats réels |
| Métier | Souscripteurs captés dans le top 10 % | référence ≈ 48,8 % | scores + résultats réels |
| Métier | Lift top 10 % | référence ≈ 4,88 | scores + résultats réels |
| Équité | Taux de sélection par sous-groupe | suivi des écarts dans le temps | scores + variables d'audit |
| Équité | FPR / FNR par sous-groupe | suivi des écarts dans le temps | scores + résultats réels |

Les seuils d'alerte proposés constituent des règles de surveillance initiales.
Ils devront être ajustés à partir du comportement réellement observé en production.

Les métriques de performance nécessitent la récupération ultérieure du résultat
réel de la campagne. Elles ne peuvent donc pas être calculées à partir des seuls
logs d'inférence.

Les variables particulièrement importantes pour la surveillance du drift sont notamment
`emp.var.rate`, `euribor3m`, `nr.employed`, `contact` et `month`, car elles figuraient
parmi les variables les plus influentes lors de l'analyse par permutation.

Une dérive de ces variables peut modifier fortement la distribution des scores même si
le service reste techniquement fonctionnel.

### 9.2 Boucle de rétroaction

Lors d'un scoring, le système conserve les informations techniques nécessaires
au suivi du modèle :

- un identifiant technique du scoring
- la date du scoring
- la version du modèle
- la probabilité estimée
- la latence de l'inférence
- le statut technique de la requête

Les données nécessaires au suivi des distributions peuvent également être
conservées dans un dispositif de monitoring dédié, sous réserve des règles de
minimisation et de conservation applicables.

À l'échelle d'une campagne, les scores des clients sont ensuite classés afin de
déterminer les clients sélectionnés selon la capacité disponible.

Après la campagne, le résultat réel est récupéré :

- client sollicité ou non
- client ayant souscrit ou non

Ces résultats permettent de rapprocher les scores initiaux de la vérité terrain
et de recalculer les métriques prédictives et métier.

Le cycle de suivi envisagé est :

```mermaid
graph LR
    A[Population à scorer]
    B[API / Gradient Boosting]
    C[Scores individuels]
    D[Classement des scores]
    E[Sélection selon capacité]
    F[Campagne]
    G[Résultats réels]
    H[Monitoring]
    I[Analyse / Réentraînement éventuel]

    A --> B
    B --> C
    C --> D
    D --> E
    E --> F
    F --> G
    C --> H
    G --> H
    H --> I
```

### 9.3 Plan de réentraînement

Le modèle n'est pas réentraîné automatiquement à fréquence fixe sans analyse
préalable.

Un réentraînement peut être envisagé lorsqu'un ou plusieurs signaux indiquent
que le modèle ou la politique de ciblage ne sont plus adaptés.

| Déclencheur | Condition proposée |
|---|---|
| Dégradation prédictive | baisse > 0.05 du ROC-AUC, de la précision, du rappel ou du F1-score |
| Dégradation métier | baisse significative de la précision top-k, du lift ou de la part de souscripteurs captés |
| Data drift | PSI > 0.20 sur une ou plusieurs variables importantes |
| Calibration | écart persistant entre probabilités estimées et fréquences observées |
| Équité | augmentation importante des écarts de sélection ou d'erreur entre sous-groupes |
| Volume de nouvelles données | quantité suffisante de nouvelles observations étiquetées |
| Évolution métier | changement important de campagne, de produit ou de population ciblée |

Une revue périodique trimestrielle est également proposée afin de vérifier la
pertinence des seuils, des variables et de la politique de ciblage.

Le dépassement d'un seuil ne déclenche pas automatiquement un remplacement du
modèle. Il déclenche d'abord une analyse de la cause.

### 9.4 Robustesse en exploitation

La robustesse du système repose sur la détection des entrées anormales, le suivi
des distributions et la surveillance de la qualité du classement produit par le
modèle.

La stratégie d'abstention initialement envisagée autour de `[0.40 ; 0.60]` n'est
pas retenue : cette zone contient au contraire une population présentant un taux
de souscription élevé.

Le contrôle en production porte donc sur d'autres mécanismes.

| Axe | Métrique / contrôle | Action proposée |
|---|---|---|
| Validation des entrées | type, valeur et catégorie autorisée | rejet HTTP 422 si entrée invalide |
| Données hors distribution | évolution importante des distributions | journaliser, analyser et éventuellement suspendre le scoring concerné |
| Data drift | PSI des variables importantes | investigation si PSI > 0.20 |
| Calibration | Brier score / comparaison score moyen et fréquence réelle | analyser ou recalibrer si dérive persistante |
| Performance prédictive | ROC-AUC, précision, rappel, F1 | analyse puis réentraînement éventuel |
| Performance métier | précision top-k, lift, souscripteurs captés | revoir modèle ou capacité de ciblage |
| Sous-groupes | taux de sélection, FPR et FNR | déclencher un audit si les écarts augmentent significativement |
| Technique | latence, erreurs API, disponibilité | investigation infrastructure / service |

L'API refuse les données qui ne respectent pas le schéma Pydantic, notamment les
catégories non autorisées.

En revanche, une donnée techniquement valide mais statistiquement inhabituelle
ne doit pas nécessairement être rejetée automatiquement. Elle constitue d'abord
un signal de drift ou de données hors distribution à analyser.

### 9.5 Synthèse amélioration continue

Le suivi de la solution repose sur plusieurs temporalités complémentaires :

- **en continu** : disponibilité, erreurs API et latence
- **quotidien / hebdomadaire** : qualité des entrées et évolution des distributions
- **après chaque campagne** : précision top-k, lift, part des souscripteurs captés,
  performances classiques et écarts entre sous-groupes
- **trimestriellement** : revue de la pertinence du modèle, des seuils d'alerte,
  de la capacité de ciblage et des règles de monitoring

Une alerte n'entraîne pas automatiquement le remplacement du modèle.

Elle déclenche d'abord une analyse permettant d'identifier la cause : dérive des
données, évolution du comportement des clients, problème de qualité des données,
dégradation du modèle ou changement du contexte métier.

Lorsqu'un réentraînement est justifié, un modèle challenger est entraîné puis
comparé au modèle champion.

Le challenger ne peut remplacer le modèle en production qu'après validation :

- de ses performances prédictives
- de ses performances métier à capacité constante
- de sa robustesse
- de son comportement sur les sous-groupes
- des tests automatisés

Cette organisation permet d'inscrire le modèle dans un cycle d'amélioration
continue sans déclencher de remplacement automatique sur la base d'un seul
indicateur.

# Annexes

## A. Glossaire métier & technique

| Terme | Définition |
|---|---|
| Classification binaire | Modèle qui prédit l'une de deux classes, ici souscription (`yes`) ou non (`no`). |
| Précision | Parmi les clients prédits positifs, proportion ayant réellement souscrit. |
| Rappel | Parmi les clients ayant réellement souscrit, proportion correctement détectée par le modèle. |
| F1-score | Moyenne harmonique entre précision et rappel. |
| ROC-AUC | Mesure de la capacité du modèle à classer les observations positives devant les négatives, indépendamment d'un seuil unique. |
| Faux positif (FP) | Client prédit susceptible de souscrire mais qui ne souscrit pas réellement. |
| Faux négatif (FN) | Client prédit peu susceptible de souscrire alors qu'il souscrit réellement. |
| Data leakage | Utilisation d'une information qui ne serait pas disponible au moment réel de la prédiction, par exemple `duration` avant l'appel. |
| Pipeline | Chaîne regroupant preprocessing et modèle afin d'appliquer exactement les mêmes transformations à l'entraînement et en production. |
| One-hot encoding | Transformation d'une variable catégorielle en plusieurs variables binaires. |
| Validation croisée | Évaluation répétée du modèle sur plusieurs découpages des données d'entraînement. |
| Abstention | Stratégie consistant à ne pas produire de décision pour certains scores jugés incertains. Cette approche a été étudiée puis abandonnée dans la solution finale. |
| Data drift | Évolution de la distribution des variables d'entrée au cours du temps. |
| Concept drift | Évolution de la relation entre les variables d'entrée et la cible. |
| PSI | Population Stability Index, indicateur utilisé pour comparer deux distributions. |
| Calibration | Correspondance entre la probabilité prédite et la fréquence réellement observée. |
| Champion / challenger | Comparaison entre le modèle actuellement en production et un nouveau modèle candidat. |
| API | Interface permettant à une application d'interroger le modèle. |
| FastAPI | Framework Python utilisé pour exposer le modèle sous forme d'API. |
| Docker | Technologie permettant d'encapsuler l'application et ses dépendances dans un environnement reproductible. |
| CI/CD | Automatisation des tests, de la construction et éventuellement du déploiement lors des modifications du projet. |

## B. Sources & bibliographie

### Données

- UCI Machine Learning Repository — **Bank Marketing Dataset**
  - Jeu de données `bank-additional-full.csv`
  - https://archive.ics.uci.edu/dataset/222/bank+marketing

### Documentation technique

- Scikit-learn — documentation officielle
  - Pipeline, preprocessing, RandomForestClassifier, validation croisée et métriques
  - https://scikit-learn.org/

- FastAPI — documentation officielle
  - https://fastapi.tiangolo.com/

- Pydantic — documentation officielle
  - https://docs.pydantic.dev/

- Streamlit — documentation officielle
  - https://docs.streamlit.io/

- Docker — documentation officielle
  - https://docs.docker.com/

- GitHub Actions — documentation officielle
  - https://docs.github.com/actions

- Pytest — documentation officielle
  - https://docs.pytest.org/

### Références réglementaires

- Règlement général sur la protection des données (RGPD) — Règlement (UE) 2016/679
- CNIL — documentation relative à la protection des données personnelles et à la prospection commerciale

## C. Décisions techniques détaillées

### Choix du scénario S3

Le scénario S1 présente les meilleures performances, mais utilise `duration`,
une variable connue uniquement après la fin de l'appel. Il ne peut donc pas être
retenu pour un ciblage réalisé avant la sollicitation.

Le scénario S3 supprime également `age`, `job`, `marital` et `education`.
Ses performances restent proches des scénarios exploitables tout en réduisant
l'utilisation directe de variables socio-démographiques potentiellement sensibles.

Il constitue donc le compromis retenu entre performance, faisabilité opérationnelle
et maîtrise du risque éthique.

### Choix du Gradient Boosting final et de la politique de ciblage

Le scénario S3 a été retenu comme périmètre principal car il exclut `duration`,
indisponible avant l'appel, ainsi que les principales variables
socio-démographiques étudiées.

Random Forest et Gradient Boosting ont ensuite été comparés plus précisément
sur ce scénario.

Une première lecture au seuil classique de 0.5 mettait en évidence des compromis
différents entre précision, rappel et F1-score. Cependant, ce seuil ne correspond
pas directement au besoin métier d'une campagne disposant d'une capacité d'appels
limitée.

Les deux modèles ont donc été comparés à capacité constante, en sélectionnant les
5 %, 10 %, 20 % et 30 % de clients ayant les scores les plus élevés.

À 10 % de capacité sur les prédictions out-of-fold du jeu d'entraînement :

- Random Forest atteint environ 50,7 % de précision, capture 45,0 % des
  souscripteurs et obtient un lift de 4,50 ;
- Gradient Boosting atteint environ 50,9 % de précision, capture 45,2 % des
  souscripteurs et obtient un lift de 4,52.

Les performances métier sont donc très proches, mais Gradient Boosting présente
également un coût CPU et une latence inférieurs dans les mesures réalisées.

Le Gradient Boosting est donc retenu comme modèle final.

La politique opérationnelle ne repose pas sur un seuil fixe de probabilité :
les clients sont classés selon leur score puis la proportion correspondant à la
capacité disponible est sélectionnée.

Dans l'hypothèse de travail étudiée, cette capacité est fixée à 10 %.

Sur le jeu de test, cette stratégie permet de cibler environ 10 % des clients,
avec un taux de souscription d'environ 55 %, de capturer environ 48,8 % de
l'ensemble des souscripteurs et d'obtenir un lift d'environ 4,88.

### Abandon de la stratégie d'abstention initiale

Une zone d'incertitude comprise entre 0.40 et 0.60 avait initialement été
envisagée afin d'éviter de prendre une décision sur les scores proches du seuil
classique de 0.5.

L'analyse complémentaire montre cependant que cette population présente un taux
réel de souscription d'environ 53 %, très supérieur au taux moyen d'environ
11,3 %.

Cette zone contient donc au contraire des profils commercialement intéressants.

La stratégie d'abstention automatique n'est pas retenue dans la solution finale.
Le modèle est utilisé comme outil de scoring et de classement, tandis que la
sélection dépend de la capacité de campagne.

### Choix de l'architecture

L'interface Streamlit ne charge pas directement le modèle : elle communique avec
une API FastAPI qui centralise la validation des entrées et l'inférence.

L'API retourne un score individuel de souscription. La politique de ciblage,
fondée sur le classement des scores et la capacité de campagne, constitue une
étape distincte réalisée à l'échelle d'une population.

Cette séparation limite le couplage entre l'interface, le modèle et la logique
de campagne et permet de faire évoluer ou remplacer chaque composant
indépendamment.

### Choix de la conteneurisation et de la CI

Docker permet de reproduire l'environnement d'exécution indépendamment de la machine
locale. GitHub Actions exécute automatiquement les tests Pytest et vérifie la construction
de l'image Docker à chaque push ou pull request.

Cette chaîne permet de détecter rapidement une régression avant un éventuel déploiement.

## D. Journal de bord

## 📅 Journal de bord – Jour 1 (30/07/2026)

### 🔧 Étapes / Actions réalisées
- Setup du projet et initialisation du dépôt Git.
- Création de deux notebooks : `journal-de-bord.ipynb` et `notebook.ipynb`
- Installation des bibliothèques nécessaires : `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`, etc.
- Analyse du problème et définition des objectifs du projet.
- Analyse du format attendu pour le notebook de journal de bord.
- Cadrage du besoin
- Identification, description et vérification de la qualité des données
- Analyse des distributions des variables et de la cible
- Etude des corrélations potentielles entre variables
- Etude des corrélations potentielles entre variables et cible
- Première analyse des variables pouvant démontrer un biais

### 📝 Observations / Difficultés
- La classe "yes" est largement minoritaire
- La variable "default" contient une proportion importante de valeurs "Unknown"
- Plusieurs variables sont fortement corrélées entre elles
- "duration" est très fortement liée à la cible, mais ne peut pas être utilisée pour prédire la cible car elle est connue seulement après l'appel
- Certains groupes démographiques présentent des taux de souscription très différents et devront etre surveillés

### ✅ Prochaines étapes
- Préparer les données pour l'entraînement du modèle
- Séparer les données d'entrainement et de test
- Entrainer un premier modèle de référence
- Comparer les différents scénarios de variables demandés
- Evaluer les modèles avec des métriques adaptée à la classe minoritaire

## 📅 Journal de bord – Jour 2 (20/08/2026)


### 🔧 Étapes / Actions réalisées
- Mise en place du split train/test
- Construction d'un pipeline de préprocessing des données
- Définition des différents scénarios de données: Complet, sans 'duration', sans variables sensibles, et scénario minimal
- Ajout d'assertions avant la modélisation sur la qualité des données
- Comparaison des grandes familles d'IA
- Choix du machine learning classique
- Sélection de trois modèles de machine learning pour la comparaison : Random Forest, Gradient Boosting et Logistic Regression
- Mise en place de l'entrainement par validation croisée


### 📝 Observations / Difficultés
- Le preprocessing doit être appris uniquement sur les données d’entraînement afin d’éviter les fuites de données.
- Les mêmes transformations doivent être utilisées pour tous les scénarios afin de conserver une comparaison équitable.
- Le déséquilibre de la cible impose d’utiliser une validation stratifiée et des métriques adaptées à la classe positive.
- Les choix d’imputation ont dû être justifiés afin de limiter leur impact sur la distribution des données.


### ✅ Prochaines étapes
- Analyser les résultats de la validation croisée.
- Comparer les performances des différents couples scénario / modèle.
- Identifier les modèles les plus pertinents à conserver pour l’évaluation détaillée.
- Étudier les métriques, matrices de confusion et éventuels biais des modèles retenus.

## 📅 Journal de bord – Jour 3 (17/09/2026)


### 🔧 Étapes / Actions réalisées

- Comparaison des modèles par validation croisée sur plusieurs métriques : précision, rappel, F1-score et ROC-AUC.
- Analyse des compromis entre précision et rappel selon les modèles et les scénarios.
- Sélection du scénario S3 comme périmètre principal d'étude, car il exclut `duration` ainsi que les variables socio-démographiques sensibles.
- Comparaison plus approfondie de la Random Forest et du Gradient Boosting.
- Test de plusieurs configurations d'hyperparamètres afin d'observer leur impact sur les performances.

### 📝 Observations / Difficultés

- Le scénario S1 obtient les meilleures performances, mais il utilise `duration`, qui n'est pas disponible avant l'appel.
- Le retrait des variables socio-démographiques dans S3 ne provoque pas de dégradation notable par rapport à S2.
- Aucun modèle ne domine sur toutes les métriques : certaines configurations favorisent davantage la précision, d'autres le rappel ou le F1-score.
- La Random Forest régularisée améliore fortement la précision et la ROC-AUC, mais au prix d'une baisse du rappel.
- Le choix du modèle doit donc être relié à l'objectif métier et ne peut pas reposer sur une seule métrique.


### ✅ Prochaines étapes

- Finaliser le choix du modèle et de la configuration d'hyperparamètres à conserver.
- Figer ce choix avant toute utilisation du jeu de test.
- Évaluer une seule fois le modèle retenu sur le test set.
- Analyser les performances finales à l'aide des métriques retenues et de la matrice de confusion.


## 📅 Journal de bord – Jour 4 (24/09/2026)


### 🔧 Étapes / Actions réalisées

- Finalisation de l’évaluation du modèle retenu sur le jeu de test.
- Analyse de la matrice de confusion et des performances finales.
- Construction du tableau d’arbitrage entre les scénarios en intégrant performance, éthique, explicabilité et contraintes opérationnelles.
- Mesure des temps d’inférence et du coût CPU des différents couples scénario / modèle.
- Analyse de l’importance des variables par permutation.
- Début de l’analyse des erreurs du modèle : faux positifs, faux négatifs, niveau de confiance et différences entre sous-groupes.

### 📝 Observations / Difficultés

- Le modèle final présente une bonne précision mais un rappel faible : il produit relativement peu de faux positifs mais manque une part importante des souscripteurs réels.
- Le scénario S3 reste le meilleur compromis entre faisabilité avant appel, performances et réduction de l’utilisation directe de variables socio-démographiques sensibles.
- Les modèles restent rapides à l’inférence ; la Random Forest est cependant plus coûteuse que les autres modèles testés.
- Les variables les plus importantes sont principalement liées au contexte économique et aux conditions de contact.
- Les faux négatifs ne sont pas uniquement concentrés près du seuil de décision : certains clients sont mal classés avec une confiance relativement forte.
- Des écarts de taux d’erreur apparaissent entre certains sous-groupes, malgré le retrait des variables sensibles du modèle.


### ✅ Prochaines étapes

- Comparer les modèles dans une logique de capacité de campagne plutôt que sur le seul seuil de 0.5.
- Vérifier si la zone d'incertitude envisagée constitue réellement une population à écarter.
- Finaliser la recommandation métier à partir des résultats de l'analyse top-k.
- Adapter ensuite l'industrialisation et le monitoring à la politique de ciblage retenue.

## 📅 Journal de bord – Jour 5 (02/10/2026)


### 🔧 Étapes / Actions réalisées

- Finalisation de l'analyse des erreurs du modèle.
- Étude des faux positifs et faux négatifs en fonction des probabilités prédites.
- Analyse des performances par sous-groupes, notamment par tranche d'âge et par profession.
- Analyse du taux de sélection par sous-groupe dans le cadre d'une politique de ciblage top 10 %.
- Étude de plusieurs zones d'incertitude autour du seuil de classification.
- Comparaison du Random Forest et du Gradient Boosting à capacité d'appels constante.
- Analyse des capacités de campagne à 5 %, 10 %, 20 % et 30 %.
- Remise en question de la stratégie d'abstention initialement envisagée entre 0.40 et 0.60.
- Révision du choix final au profit du Gradient Boosting et d'une politique fondée sur le classement des scores.


### 📝 Observations / Difficultés

- Le seuil classique de 0.5 produit un modèle très sélectif et ne correspond pas directement à une contrainte de capacité d'appels.
- Random Forest et Gradient Boosting présentent des performances métier très proches lorsque la comparaison est réalisée à capacité constante.
- Gradient Boosting présente un léger avantage sur les métriques top-k ainsi qu'un coût CPU et une latence inférieurs.
- La zone `[0.40 ; 0.60]`, initialement envisagée comme zone d'abstention, présente en réalité un taux de souscription très supérieur à la moyenne.
- La stratégie d'abstention a donc été abandonnée après analyse des résultats.
- Les taux d'erreur et de sélection diffèrent entre certains groupes malgré le retrait direct de `age` et `job`, ce qui confirme la nécessité d'un suivi des biais indirects.
- Le choix final a donc évolué à partir des résultats obtenus et non d'une hypothèse fixée a priori.

### ✅ Prochaines étapes

- Persister le pipeline final.
- Exposer le modèle via une API.
- Mettre en place une interface utilisateur simple.
- Documenter l'architecture cible.
- Préparer la conteneurisation et les tests automatisés.


## 📅 Journal de bord – Jour 6 (05/10/2026)


### 🔧 Étapes / Actions réalisées

- Persistance du pipeline final de preprocessing et de Gradient Boosting au format `joblib`.
- Mise en place d'une API FastAPI.
- Création des routes `/health` et `/predict`.
- Transformation de `/predict` en endpoint de scoring retournant une probabilité individuelle de souscription.
- Suppression de la logique d'abstention de l'API.
- Ajout de logs applicatifs pour le statut, la probabilité, la latence et la version du modèle.
- Renforcement de la validation des données d'entrée avec Pydantic.
- Validation explicite des catégories autorisées avec `Literal`.
- Ajout de contraintes simples sur `campaign`, `pdays` et `previous`.
- Création d'une interface Streamlit permettant de scorer un profil client.
- Séparation entre le scoring individuel et la politique top 10 % réalisée à l'échelle d'une campagne.
- Mise à jour du schéma d'architecture de la solution.


### 📝 Observations / Difficultés

- Les noms de variables contenant des points, comme `emp.var.rate`, nécessitent l'utilisation d'alias dans les modèles Pydantic.
- La politique top 10 % ne peut pas être appliquée à partir d'un seul client : elle nécessite de scorer puis de classer l'ensemble de la population de campagne.
- L'API doit donc rester un service de scoring et ne pas prendre seule la décision de ciblage.
- La validation des seuls types ne suffit pas : une catégorie inexistante comme `month="foo"` était auparavant acceptée par le pipeline à cause de `OneHotEncoder(handle_unknown="ignore")`.
- La validation Pydantic a été renforcée afin de rejeter ce type d'entrée avant l'inférence.
- Le logging est limité aux informations techniques nécessaires au suivi, sans journaliser directement les données personnelles du profil.


### ✅ Prochaines étapes

- Conteneuriser l'API et l'interface.
- Ajouter des tests automatisés sur les principaux comportements de l'API.
- Mettre en place une pipeline d'intégration continue.
- Vérifier que la construction Docker fonctionne dans la CI.


## 📅 Journal de bord – Jour 7 (06/10/2026)


### 🔧 Étapes / Actions réalisées

- Mise en place de Docker pour l'API et de Docker Compose pour l'orchestration des services.
- Création et mise à jour des tests automatisés avec `pytest`.
- Tests de la route `/health`, du scoring via `/predict`, du placeholder `/train` et de la validation des entrées.
- Ajout de tests vérifiant le rejet des catégories invalides et des valeurs numériques incohérentes.
- Adaptation des tests à la suppression de la logique d'abstention.
- Correction des mocks de `predict_proba` afin qu'ils reproduisent le format NumPy renvoyé par Scikit-learn.
- Mise en place d'une pipeline GitHub Actions.
- Exécution automatique des tests et de la construction de l'image Docker à chaque push ou pull request.
- Correction de l'absence de l'artefact `pipeline.joblib` dans le dépôt afin de permettre l'exécution de la CI.
- Validation d'un workflow GitHub Actions réussi.
- Révision du plan de monitoring pour suivre les métriques techniques, le drift, les performances prédictives, les performances top-k et les écarts entre sous-groupes.
- Définition de la boucle de rétroaction et du mécanisme champion / challenger pour les futurs réentraînements.


### 📝 Observations / Difficultés

- Les premiers tests GitHub Actions échouaient car `pipeline.joblib` était ignoré par Git et n'était donc pas disponible dans l'environnement CI.
- Le versionnement explicite de l'artefact final a permis de rendre le projet autonome pour la démonstration.
- Un problème d'import du package `api` lors de l'exécution locale de `pytest` a été corrigé afin de rendre les tests reproductibles depuis la racine du projet.
- Les métriques comme le F1-score, la précision top-k ou le lift nécessitent de récupérer la vérité terrain après la campagne et ne peuvent pas être calculées à partir des seuls logs API.
- Le monitoring doit donc combiner des indicateurs immédiatement disponibles, comme la latence et les erreurs API, avec des métriques différées issues des résultats métier.
- La révision du modèle et de la politique de ciblage a nécessité de réaligner l'API, l'interface, les tests, le monitoring et la documentation.
- Un réentraînement automatique sans validation présente un risque ; une comparaison champion / challenger est préférable avant remplacement du modèle en production.

### ✅ Prochaines étapes

- Nettoyer et relire le notebook principal.
- Vérifier les TODO et supprimer les éléments non implémentés présentés comme acquis.
- Finaliser les annexes et les sources.
- Fusionner le journal de bord avec le notebook principal.
- Exécuter un `Restart & Run All` sur le notebook final.
- Préparer la version définitive à remettre pour la certification.
